In [ ]:
# Cell 1: Setup — install deps, mount Drive, check GPU
!pip install -q torch transformers datasets sentencepiece pyyaml seqeval conllu accelerate huggingface_hub

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Clone repo + download data
import os, subprocess

if not os.path.exists("/content/kniv-nlp-models"):
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/rustic-ai/kniv-nlp-models.git",
    ], cwd="/content")
    print("Repo cloned.")

from huggingface_hub import snapshot_download
snapshot_download(
    "dragonscale-ai/kniv-corpus-en",
    repo_type="dataset",
    allow_patterns="prepared/kniv-deberta-cascade/*",
    local_dir="/content/kniv-data",
)
print("Data downloaded.")

os.chdir("/content/kniv-nlp-models")
os.makedirs("data/prepared", exist_ok=True)
if not os.path.exists("data/prepared/kniv-deberta-cascade"):
    os.symlink(
        "/content/kniv-data/prepared/kniv-deberta-cascade",
        "data/prepared/kniv-deberta-cascade",
    )
print(f"Working dir: {os.getcwd()}")
print(f"SRL silver data: {os.path.exists('data/prepared/kniv-deberta-cascade/srl_silver_dannashao.json')}")
!ls -lh data/prepared/kniv-deberta-cascade/srl_*.json

In [ ]:
# Cell 3: Phase 1 — SRL encoder fine-tuning (Shi & Lin style)
import subprocess, os
os.chdir("/content/kniv-nlp-models")
subprocess.check_call(["git", "pull"])
print("Code updated.")

DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
os.makedirs(DRIVE_OUT, exist_ok=True)

!python models/kniv-deberta-cascade-large-nlp-en/train_srl_encoder.py \
    --encoder microsoft/deberta-v3-large \
    --train-file srl_silver_dannashao.json \
    --output-dir {DRIVE_OUT} \
    --max-examples 200000 \
    --epochs 3 \
    --batch-size 32 \
    --encoder-lr 2e-5 \
    --head-lr 1e-4

In [ ]:
# Cell 4: Evaluate Phase 1 model on silver data sample (to check if it learned the silver distribution)
import json, random, torch, sys
from pathlib import Path
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")

# Load a held-out silver sample (different from training subsample)
with open("data/prepared/kniv-deberta-cascade/srl_silver_dannashao.json") as f:
    all_silver = json.load(f)

# Use examples 200K-201K (outside the training subsample which used first 200K with seed 42)
random.seed(99)
silver_eval = random.sample(all_silver[200000:], min(1236, len(all_silver) - 200000))
print(f"Silver eval: {len(silver_eval)} examples")

# Also check gold dev
with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = json.load(f)
gold_dev = [ex for ex in gold_dev if "predicate_idx" in ex]
print(f"Gold dev: {len(gold_dev)} examples")

# Load model
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer

DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
device = torch.device("cuda")

model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/best/model.pt", weights_only=True)
model.load_state_dict(state)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/best")

# Evaluate on both
from torch.utils.data import DataLoader

silver_ds = SRLDataset(silver_eval, tokenizer)
silver_loader = DataLoader(silver_ds, batch_size=64, collate_fn=collate_fn)

gold_ds = SRLDataset(gold_dev, tokenizer)
gold_loader = DataLoader(gold_ds, batch_size=64, collate_fn=collate_fn)

print("\n=== Silver eval (held-out) ===")
f1_silver, report_silver = evaluate(model, silver_loader, device)
print(f"SRL F1: {f1_silver:.4f}")
print(report_silver)

print("\n=== Gold dev (PropBank) ===")
f1_gold, report_gold = evaluate(model, gold_loader, device)
print(f"SRL F1: {f1_gold:.4f}")
print(report_gold)

In [ ]:
# Cell 5: Fine-tune Phase 1 model on gold PropBank data (15K)
# The encoder already has predicate awareness (94.8% on silver).
# Now adapt to gold PropBank label conventions.

import json, random, torch, sys
from pathlib import Path
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")

from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS, TAG2ID
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load the Phase 1 silver-trained model
print("Loading Phase 1 model...")
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/best")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/best/model.pt", weights_only=True)
model.load_state_dict(state)
model = model.to(device)
print("Phase 1 model loaded.")

# Load gold data
with open("data/prepared/kniv-deberta-cascade/srl_full_train.json") as f:
    gold_all = json.load(f)
gold_all = [ex for ex in gold_all if "predicate_idx" in ex and any(t not in ("O","V") for t in ex["srl_tags"])]
print(f"Gold data: {len(gold_all)} examples")

# Subsample to 15K
random.seed(42)
gold_train = random.sample(gold_all, min(15000, len(gold_all)))
print(f"Gold train: {len(gold_train)} examples")

# Dev
with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = json.load(f)
gold_dev = [ex for ex in gold_dev if "predicate_idx" in ex]

train_ds = SRLDataset(gold_train, tokenizer)
dev_ds = SRLDataset(gold_dev, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_ds, batch_size=64, collate_fn=collate_fn)

# Lower LR since model is already trained — this is adaptation, not training from scratch
encoder_params = [p for n, p in model.named_parameters() if "deberta" in n or "pred_embedding" in n]
head_params = [p for n, p in model.named_parameters() if "classifier" in n]
optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": 5e-6},   # gentle encoder adaptation
    {"params": head_params, "lr": 5e-5},       # head adapts faster
], weight_decay=0.01)

epochs = 5
total_steps = epochs * len(train_loader)
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print(f"\nGold adaptation: {epochs} epochs, {len(train_loader)} batches/epoch")
print(f"Encoder LR: 5e-6, Head LR: 5e-5\n")

best_f1 = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["predicate_idx"].to(device),
        )
        loss = loss_fn(logits.view(-1, len(SRL_TAGS)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    f1, report = evaluate(model, dev_loader, device)
    print(f"\nEpoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f}, SRL F1: {f1:.4f}")
    print(report)

    if f1 > best_f1:
        best_f1 = f1
        model.save_full(f"{DRIVE_OUT}/gold_best")
        model.save_encoder(f"{DRIVE_OUT}/gold_best_encoder")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/gold_best")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/gold_best_encoder")
        print(f"  * New best F1={f1:.4f}")

print(f"\nGold adaptation complete. Best SRL F1: {best_f1:.4f}")

In [ ]:
# Cell 6: Evaluate gold-adapted model on both silver and gold dev
import json, random, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load gold-adapted model
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best/model.pt", weights_only=True)
model.load_state_dict(state)
model = model.to(device)

# Silver held-out
with open("data/prepared/kniv-deberta-cascade/srl_silver_dannashao.json") as f:
    all_silver = json.load(f)
random.seed(99)
silver_eval = random.sample(all_silver[200000:], 1236)

# Gold dev
with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = json.load(f)
gold_dev = [ex for ex in gold_dev if "predicate_idx" in ex]

silver_loader = DataLoader(SRLDataset(silver_eval, tokenizer), batch_size=64, collate_fn=collate_fn)
gold_loader = DataLoader(SRLDataset(gold_dev, tokenizer), batch_size=64, collate_fn=collate_fn)

print("=== Silver held-out ===")
f1_s, rep_s = evaluate(model, silver_loader, device)
print(f"SRL F1: {f1_s:.4f}\n{rep_s}")

print("=== Gold dev ===")
f1_g, rep_g = evaluate(model, gold_loader, device)
print(f"SRL F1: {f1_g:.4f}\n{rep_g}")

In [ ]:
# Cell 7: Phase 1b — Train from scratch on pure gold PropBank (no silver)
# Test: does a vanilla DeBERTa + gold-only match or beat silver-pretrained + gold?

import json, random, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Fresh DeBERTa — no silver pre-training
print("Loading fresh DeBERTa-v3-large (no silver)...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float().to(device)
print("Fresh model loaded.")

# Load ALL gold data (no silver)
with open("data/prepared/kniv-deberta-cascade/srl_full_train.json") as f:
    gold_all = json.load(f)
gold_all = [ex for ex in gold_all if "predicate_idx" in ex and any(t not in ("O","V") for t in ex["srl_tags"])]
print(f"Gold data: {len(gold_all)} examples")

with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = json.load(f)
gold_dev = [ex for ex in gold_dev if "predicate_idx" in ex]

train_ds = SRLDataset(gold_all, tokenizer)
dev_ds = SRLDataset(gold_dev, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_ds, batch_size=64, collate_fn=collate_fn)

# Standard fine-tuning LR
encoder_params = [p for n, p in model.named_parameters() if "deberta" in n or "pred_embedding" in n]
head_params = [p for n, p in model.named_parameters() if "classifier" in n]
optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": 2e-5},
    {"params": head_params, "lr": 1e-4},
], weight_decay=0.01)

epochs = 3
total_steps = epochs * len(train_loader)
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print(f"\nPure gold training: {epochs} epochs, {len(train_loader)} batches/epoch")
print(f"Encoder LR: 2e-5, Head LR: 1e-4\n")

best_f1 = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["predicate_idx"].to(device),
        )
        loss = loss_fn(logits.view(-1, len(SRL_TAGS)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    f1, report = evaluate(model, dev_loader, device)
    print(f"\nEpoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f}, SRL F1: {f1:.4f}")
    print(report)

    if f1 > best_f1:
        best_f1 = f1
        model.save_full(f"{DRIVE_OUT}/pure_gold_best")
        model.save_encoder(f"{DRIVE_OUT}/pure_gold_best_encoder")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/pure_gold_best")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/pure_gold_best_encoder")
        print(f"  * New best F1={f1:.4f}")

print(f"\nPure gold training complete. Best SRL F1: {best_f1:.4f}")

In [ ]:
# Cell 8: Phase 1c — Continue gold adaptation on silver-pretrained model
# Resume from the gold_best checkpoint, train 5 more epochs on 35K gold
# (held-out 35K from the 67K, different from the 15K used in Cell 5)

import json, random, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load the gold-adapted model (silver pretrained → 15K gold → 0.653)
print("Loading gold-adapted model (F1=0.653)...")
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best/model.pt", weights_only=True)
model.load_state_dict(state)
model = model.to(device)
print("Model loaded.")

# Load gold data — sample 35K (different seed than Cell 5's 15K)
with open("data/prepared/kniv-deberta-cascade/srl_full_train.json") as f:
    gold_all = json.load(f)
gold_all = [ex for ex in gold_all if "predicate_idx" in ex and any(t not in ("O","V") for t in ex["srl_tags"])]

random.seed(123)  # different seed → different 35K subset
gold_train = random.sample(gold_all, min(35000, len(gold_all)))
print(f"Gold train: {len(gold_train)} examples")

# Dev
with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = json.load(f)
gold_dev = [ex for ex in gold_dev if "predicate_idx" in ex]

train_ds = SRLDataset(gold_train, tokenizer)
dev_ds = SRLDataset(gold_dev, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_ds, batch_size=64, collate_fn=collate_fn)

# Continuation LR — lower since already adapted
encoder_params = [p for n, p in model.named_parameters() if "deberta" in n or "pred_embedding" in n]
head_params = [p for n, p in model.named_parameters() if "classifier" in n]
optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": 3e-6},
    {"params": head_params, "lr": 3e-5},
], weight_decay=0.01)

epochs = 5
total_steps = epochs * len(train_loader)
warmup_steps = int(total_steps * 0.05)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print(f"\nContinued gold training: {epochs} epochs, {len(train_loader)} batches/epoch")
print(f"Encoder LR: 3e-6, Head LR: 3e-5 (continuation)\n")

best_f1 = 0.653  # starting point
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["predicate_idx"].to(device),
        )
        loss = loss_fn(logits.view(-1, len(SRL_TAGS)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    f1, report = evaluate(model, dev_loader, device)
    print(f"\nEpoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f}, SRL F1: {f1:.4f}")
    print(report)

    if f1 > best_f1:
        best_f1 = f1
        model.save_full(f"{DRIVE_OUT}/gold_best")
        model.save_encoder(f"{DRIVE_OUT}/gold_best_encoder")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/gold_best")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/gold_best_encoder")
        print(f"  * New best F1={f1:.4f}")

print(f"\nContinued training complete. Best SRL F1: {best_f1:.4f}")

In [ ]:
# Cell 9: Download CoNLL-2012 benchmark
import subprocess, os
os.chdir("/content/kniv-nlp-models")
subprocess.check_call(["git", "pull"])

# Need older datasets version that supports loading scripts
!pip install -q "datasets==2.20.0"

# Convert CoNLL-2012 test set
!python models/kniv-deberta-cascade-large-nlp-en/prepare_conll2012_eval.py \
    --split test --output data/prepared/kniv-deberta-cascade/conll2012_test.json

# Also convert validation set
!python models/kniv-deberta-cascade-large-nlp-en/prepare_conll2012_eval.py \
    --split validation --output data/prepared/kniv-deberta-cascade/conll2012_dev.json

In [ ]:
# Cell 10: Benchmark with fixed eval (V excluded from scoring)
import subprocess, json, torch, sys
subprocess.check_call(["git", "pull"], cwd="/content/kniv-nlp-models")

sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import importlib, train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best/model.pt", weights_only=True)
model.load_state_dict(state)
model = model.to(device)

# CoNLL-2012 test
with open("data/prepared/kniv-deberta-cascade/conll2012_test.json") as f:
    conll_test = [ex for ex in json.load(f) if "predicate_idx" in ex]
loader = DataLoader(SRLDataset(conll_test, tokenizer), batch_size=64, collate_fn=collate_fn)
f1, report = evaluate(model, loader, device)
print(f"=== CoNLL-2012 Test — SRL F1: {f1:.4f} (SOTA: 87.67) ===")
print(report)

# PropBank EWT dev
with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]
loader2 = DataLoader(SRLDataset(gold_dev, tokenizer), batch_size=64, collate_fn=collate_fn)
f1_g, report_g = evaluate(model, loader2, device)
print(f"=== PropBank EWT Dev — SRL F1: {f1_g:.4f} ===")
print(report_g)

In [ ]:
# Cell 11: Continue gold training — 5 more epochs on 40K gold
import json, random, torch, sys, importlib
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load current best
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best/model.pt", weights_only=True)
model.load_state_dict(state)
model = model.to(device)
print("Loaded gold_best model")

# 40K gold, new seed
with open("data/prepared/kniv-deberta-cascade/srl_full_train.json") as f:
    gold_all = json.load(f)
gold_all = [ex for ex in gold_all if "predicate_idx" in ex and any(t not in ("O","V") for t in ex["srl_tags"])]
random.seed(777)
gold_train = random.sample(gold_all, min(40000, len(gold_all)))
print(f"Gold train: {len(gold_train)}")

with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]

train_loader = DataLoader(SRLDataset(gold_train, tokenizer), batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(SRLDataset(gold_dev, tokenizer), batch_size=64, collate_fn=collate_fn)

encoder_params = [p for n, p in model.named_parameters() if "deberta" in n or "pred_embedding" in n]
head_params = [p for n, p in model.named_parameters() if "classifier" in n]
optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": 2e-6},
    {"params": head_params, "lr": 2e-5},
], weight_decay=0.01)

epochs = 5
total_steps = epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.05), total_steps)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print(f"\n5 more epochs, {len(train_loader)} batches/epoch, LR 2e-6/2e-5\n")

best_f1 = 0.681
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["predicate_idx"].to(device))
        loss = loss_fn(logits.view(-1, len(SRL_TAGS)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    f1, report = evaluate(model, dev_loader, device)
    print(f"\nEpoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, SRL F1: {f1:.4f}")
    print(report)
    if f1 > best_f1:
        best_f1 = f1
        model.save_full(f"{DRIVE_OUT}/gold_best")
        model.save_encoder(f"{DRIVE_OUT}/gold_best_encoder")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/gold_best")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/gold_best_encoder")
        print(f"  * New best F1={f1:.4f}")

print(f"\nDone. Best SRL F1: {best_f1:.4f}")

In [ ]:
# Cell 12: Compare greedy vs Viterbi decoding on current best model
import subprocess, json, torch, sys, importlib
subprocess.check_call(["git", "pull"], cwd="/content/kniv-nlp-models")
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best/model.pt", weights_only=True)
model.load_state_dict(state)
model = model.to(device)

with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]
dev_loader = DataLoader(SRLDataset(gold_dev, tokenizer), batch_size=64, collate_fn=collate_fn)

print("=== Greedy (argmax) ===")
f1_greedy, _ = evaluate(model, dev_loader, device, use_viterbi=False)
print(f"PropBank Dev F1: {f1_greedy:.4f}\n")

print("=== Viterbi decoding ===")
f1_viterbi, report = evaluate(model, dev_loader, device, use_viterbi=True)
print(f"PropBank Dev F1: {f1_viterbi:.4f}")
print(f"Improvement: +{f1_viterbi - f1_greedy:.4f}")
print(report)

In [ ]:
# Check what's saved on Drive
import os
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
for d in sorted(os.listdir(DRIVE_OUT)):
    path = f"{DRIVE_OUT}/{d}"
    if os.path.isdir(path):
        files = os.listdir(path)
        size = sum(os.path.getsize(f"{path}/{f}") for f in files) / 1e6
        print(f"  {d}/ — {len(files)} files, {size:.0f} MB")

In [ ]:
# Cell 14: CoNLL-2012 benchmark with Viterbi decoding
import json, torch, sys, importlib
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best/model.pt", weights_only=True)
model.load_state_dict(state)
model = model.to(device)

with open("data/prepared/kniv-deberta-cascade/conll2012_test.json") as f:
    conll_test = [ex for ex in json.load(f) if "predicate_idx" in ex]
print(f"CoNLL-2012 test: {len(conll_test)} examples")

loader = DataLoader(SRLDataset(conll_test, tokenizer), batch_size=64, collate_fn=collate_fn)

print("\n=== CoNLL-2012 Test + Viterbi (SOTA: 87.67) ===")
f1, report = evaluate(model, loader, device, use_viterbi=True)
print(f"SRL F1: {f1:.4f}")
print(report)

In [ ]:
# Cell 15: Extend silver SRL spans using dep trees
import subprocess, os
os.chdir("/content/kniv-nlp-models")
subprocess.check_call(["git", "pull"])

# Download annotated corpus from HuggingFace
from huggingface_hub import snapshot_download
snapshot_download(
    "dragonscale-ai/kniv-corpus-en",
    repo_type="dataset",
    allow_patterns="annotated/*",
    local_dir="/content/kniv-corpus",
)
os.makedirs("corpus/output", exist_ok=True)
if not os.path.exists("corpus/output/annotated"):
    os.symlink("/content/kniv-corpus/annotated", "corpus/output/annotated")
print(f"Corpus: {os.listdir('corpus/output/annotated/')}")

# Ensure silver data
import shutil
SILVER_SRC = "/content/drive/MyDrive/kniv-models/srl_silver/srl_silver_dannashao.json"
SILVER_DST = "data/prepared/kniv-deberta-cascade/srl_silver_dannashao.json"
if not os.path.exists(SILVER_DST):
    shutil.copy2(SILVER_SRC, SILVER_DST)

!python models/kniv-deberta-cascade-large-nlp-en/extend_srl_spans.py \
    --silver srl_silver_dannashao.json \
    --corpus-dir corpus/output/annotated

In [ ]:
# Cell 16: Backup current gold_best model (0.738 Viterbi)
import shutil, os

DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
BACKUP = f"{DRIVE_OUT}/gold_best_v1_0738"

if not os.path.exists(BACKUP):
    shutil.copytree(f"{DRIVE_OUT}/gold_best", BACKUP)
    print(f"Backed up to {BACKUP}")
    print(f"Files: {os.listdir(BACKUP)}")
else:
    print(f"Backup already exists: {BACKUP}")

In [ ]:
# Cell 17: Filter extended silver to only examples with multi-word spans
import json

with open("data/prepared/kniv-deberta-cascade/srl_silver_extended.json") as f:
    extended = json.load(f)
print(f"Total extended: {len(extended):,}")

# Keep only examples that have at least one I- tag (multi-word span)
multi_word = [ex for ex in extended if any(t.startswith("I-") for t in ex["srl_tags"])]
print(f"With multi-word spans: {len(multi_word):,} ({100*len(multi_word)/len(extended):.1f}%)")

# Stats
from collections import Counter
span_lens = []
for ex in multi_word:
    cur_len = 0
    for t in ex["srl_tags"]:
        if t.startswith("B-"):
            if cur_len > 0: span_lens.append(cur_len)
            cur_len = 1
        elif t.startswith("I-"):
            cur_len += 1
        else:
            if cur_len > 0: span_lens.append(cur_len)
            cur_len = 0
    if cur_len > 0: span_lens.append(cur_len)

multi_spans = sum(1 for l in span_lens if l > 1)
print(f"\nSpan stats:")
print(f"  Total spans: {len(span_lens):,}")
print(f"  Mean length: {sum(span_lens)/len(span_lens):.2f}")
print(f"  Multi-word: {multi_spans:,} ({100*multi_spans/len(span_lens):.1f}%)")

len_dist = Counter(span_lens)
for l in sorted(len_dist.keys())[:8]:
    print(f"  {l}-tok: {len_dist[l]:,} ({100*len_dist[l]/len(span_lens):.1f}%)")

# Save
with open("data/prepared/kniv-deberta-cascade/srl_silver_multiword.json", "w") as f:
    json.dump(multi_word, f)
print(f"\nSaved {len(multi_word):,} examples to srl_silver_multiword.json")

In [ ]:
# Cell 18: Phase 1 v2 — Silver pre-training with extended multi-word spans
# Fresh DeBERTa, 730K multi-word silver, larger batch, lower LR, 3 epochs
# Then gold adaptation on top

import json, random, torch, sys, importlib
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Fresh encoder
print("Loading fresh DeBERTa-v3-large...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float().to(device)

# Load multi-word silver data — subsample to 200K for speed
with open("data/prepared/kniv-deberta-cascade/srl_silver_multiword.json") as f:
    silver = json.load(f)
random.seed(42)
silver_train = random.sample(silver, min(200000, len(silver)))
print(f"Silver train: {len(silver_train):,} (multi-word spans)")

with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]

train_loader = DataLoader(SRLDataset(silver_train, tokenizer), batch_size=64, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(SRLDataset(gold_dev, tokenizer), batch_size=64, collate_fn=collate_fn)

encoder_params = [p for n, p in model.named_parameters() if "deberta" in n or "pred_embedding" in n]
head_params = [p for n, p in model.named_parameters() if "classifier" in n]
optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": 1e-5},
    {"params": head_params, "lr": 5e-5},
], weight_decay=0.01)

epochs = 3
total_steps = epochs * len(train_loader)
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print(f"\nSilver v2: {epochs} epochs, {len(train_loader)} batches/epoch, batch_size=64")
print(f"Encoder LR: 1e-5, Head LR: 5e-5\n")

best_f1 = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["predicate_idx"].to(device))
        loss = loss_fn(logits.view(-1, len(SRL_TAGS)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if (step + 1) % 200 == 0:
            print(f"  step={epoch*len(train_loader)+step+1} loss={loss.item():.4f} lr={scheduler.get_last_lr()[0]:.2e}/{scheduler.get_last_lr()[1]:.2e}", flush=True)

    f1, report = evaluate(model, dev_loader, device, use_viterbi=True)
    print(f"\nEpoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, SRL F1 (VD): {f1:.4f}")
    print(report)
    if f1 > best_f1:
        best_f1 = f1
        model.save_full(f"{DRIVE_OUT}/silver_v2_best")
        model.save_encoder(f"{DRIVE_OUT}/silver_v2_best_encoder")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/silver_v2_best")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/silver_v2_best_encoder")
        print(f"  * New best F1={f1:.4f}")

print(f"\nSilver v2 complete. Best SRL F1: {best_f1:.4f}")

In [ ]:
# Cell 19: Frozen encoder+marker, train ONLY MLP head on 20K gold
# Encoder has predicate awareness (94.8% silver). Head learns gold labels.
import json, random, torch, sys, importlib
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load silver-pretrained model (the original Phase 1 best)
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/best")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/best/model.pt", weights_only=True)
model.load_state_dict(state)
model = model.to(device)

# Freeze encoder + pred_embedding, only train classifier
frozen = 0
trainable = 0
for name, param in model.named_parameters():
    if "classifier" in name:
        param.requires_grad = True
        trainable += param.numel()
    else:
        param.requires_grad = False
        frozen += param.numel()
print(f"Frozen: {frozen:,} params | Trainable: {trainable:,} params (classifier only)")

# 20K gold
with open("data/prepared/kniv-deberta-cascade/srl_full_train.json") as f:
    gold_all = json.load(f)
gold_all = [ex for ex in gold_all if "predicate_idx" in ex and any(t not in ("O","V") for t in ex["srl_tags"])]
random.seed(42)
gold_train = random.sample(gold_all, min(20000, len(gold_all)))
print(f"Gold train: {len(gold_train):,}")

with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]

train_loader = DataLoader(SRLDataset(gold_train, tokenizer), batch_size=128, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(SRLDataset(gold_dev, tokenizer), batch_size=128, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3, weight_decay=0.01,
)

epochs = 10
total_steps = epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print(f"\nFrozen encoder, head-only training: {epochs} epochs, {len(train_loader)} batches/epoch")
print(f"Head LR: 1e-3 (high, small head)\n")

best_f1 = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["predicate_idx"].to(device))
        loss = loss_fn(logits.view(-1, len(SRL_TAGS)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    f1, report = evaluate(model, dev_loader, device, use_viterbi=True)
    print(f"\nEpoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, SRL F1 (VD): {f1:.4f}")
    print(report)
    if f1 > best_f1:
        best_f1 = f1
        model.save_full(f"{DRIVE_OUT}/frozen_head_best")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/frozen_head_best")
        print(f"  * New best F1={f1:.4f}")
    elif f1 < best_f1 - 0.02:
        print(f"  Degrading, stopping early")
        break

print(f"\nDone. Best SRL F1: {best_f1:.4f}")

In [ ]:
# Cell 21: POS head — standard encoding, no pred_embedding
import json, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import importlib, train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load gold-adapted SRL model
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best_v1_0738")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best_v1_0738/model.pt", weights_only=True)
model.load_state_dict(state)

# Add POS head
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
H = model.deberta.config.hidden_size
model.pos_head = nn.Linear(H, len(pos_labels))
model = model.to(device)

# Freeze everything except pos_head
for name, param in model.named_parameters():
    param.requires_grad = "pos_head" in name
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (pos_head only)")

# Standard forward — NO pred_embedding, just encoder directly
def forward_pos(input_ids, attention_mask):
    hidden = model.deberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
    return model.pos_head(hidden)

# POS dataset — simple, no predicate_idx needed
class POSDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples = examples
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_length = max_length
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        encoding = self.tokenizer(ex["words"], is_split_into_words=True,
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt")
        word_ids = encoding.word_ids()
        aligned, prev = [], None
        for wid in word_ids:
            if wid is None: aligned.append(-100)
            elif wid != prev:
                tag = ex["pos_tags"][wid] if wid < len(ex["pos_tags"]) else "X"
                aligned.append(self.label_map.get(tag, 0))
            else: aligned.append(-100)
            prev = wid
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(aligned, dtype=torch.long),
        }

def collate(feats):
    return {k: torch.stack([f[k] for f in feats]) for k in feats[0]}

with open("data/prepared/kniv-deberta-cascade/ud_train.json") as f:
    ud_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f:
    ud_dev = json.load(f)

pos_map = {l: i for i, l in enumerate(pos_labels)}
train_loader = DataLoader(POSDataset(ud_train, tokenizer, pos_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(POSDataset(ud_dev, tokenizer, pos_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.1), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print(f"POS training (no pred marker): {epochs} epochs, {len(train_loader)} batches/epoch\n")

best_acc = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = forward_pos(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(pos_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        total_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in dev_loader:
            logits = forward_pos(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            mask = batch["labels"] != -100
            correct += (preds[mask] == batch["labels"][mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, POS Acc: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.pos_head.state_dict(), f"{DRIVE_OUT}/pos_head_best.pt")
        print(f"  * New best Acc={acc:.4f}")

print(f"\nBest POS Acc: {best_acc:.4f}")

In [ ]:
# Cell 22: Continue POS — reload best head, higher LR, 10 more epochs
import json, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Reload model with best POS head
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best_v1_0738")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best_v1_0738/model.pt", weights_only=True)
model.load_state_dict(state)

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
H = model.deberta.config.hidden_size
model.pos_head = nn.Linear(H, len(pos_labels))
# Load the best POS head weights from previous run
pos_state = torch.load(f"{DRIVE_OUT}/pos_head_best.pt", weights_only=True)
model.pos_head.load_state_dict(pos_state)
model = model.to(device)

for name, param in model.named_parameters():
    param.requires_grad = "pos_head" in name

def forward_pos(input_ids, attention_mask):
    hidden = model.deberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
    return model.pos_head(hidden)

class POSDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples = examples
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_length = max_length
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        encoding = self.tokenizer(ex["words"], is_split_into_words=True,
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt")
        word_ids = encoding.word_ids()
        aligned, prev = [], None
        for wid in word_ids:
            if wid is None: aligned.append(-100)
            elif wid != prev:
                tag = ex["pos_tags"][wid] if wid < len(ex["pos_tags"]) else "X"
                aligned.append(self.label_map.get(tag, 0))
            else: aligned.append(-100)
            prev = wid
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(aligned, dtype=torch.long),
        }

def collate(feats):
    return {k: torch.stack([f[k] for f in feats]) for k in feats[0]}

with open("data/prepared/kniv-deberta-cascade/ud_train.json") as f:
    ud_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f:
    ud_dev = json.load(f)

pos_map = {l: i for i, l in enumerate(pos_labels)}
train_loader = DataLoader(POSDataset(ud_train, tokenizer, pos_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(POSDataset(ud_dev, tokenizer, pos_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-3)
epochs = 10
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.05), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print(f"POS continued: {epochs} epochs, LR 5e-3, starting from 0.807\n")

best_acc = 0.807
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = forward_pos(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(pos_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        total_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in dev_loader:
            logits = forward_pos(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            mask = batch["labels"] != -100
            correct += (preds[mask] == batch["labels"][mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, POS Acc: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.pos_head.state_dict(), f"{DRIVE_OUT}/pos_head_best.pt")
        print(f"  * New best Acc={acc:.4f}")

print(f"\nBest POS Acc: {best_acc:.4f}")

In [ ]:
# Cell 23: POS — unfreeze encoder at 1e-7, head at 1e-3, from 0.856
import json, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best_v1_0738")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best_v1_0738/model.pt", weights_only=True)
model.load_state_dict(state)

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
model.pos_head = nn.Linear(model.deberta.config.hidden_size, len(pos_labels))
model.pos_head.load_state_dict(torch.load(f"{DRIVE_OUT}/pos_head_best.pt", weights_only=True))
model = model.to(device)

encoder_params, head_params = [], []
for n, p in model.named_parameters():
    p.requires_grad = True
    if "pos_head" in n:
        head_params.append(p)
    else:
        encoder_params.append(p)
print(f"Encoder: {sum(p.numel() for p in encoder_params):,} | Head: {sum(p.numel() for p in head_params):,}")

def forward_pos(input_ids, attention_mask):
    return model.pos_head(model.deberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state)

class POSDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["pos_tags"][wid] if wid < len(ex["pos_tags"]) else "X", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}

with open("data/prepared/kniv-deberta-cascade/ud_train.json") as f: ud_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f: ud_dev = json.load(f)
pos_map = {l: i for i, l in enumerate(pos_labels)}
train_loader = DataLoader(POSDataset(ud_train, tokenizer, pos_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(POSDataset(ud_dev, tokenizer, pos_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([{"params": encoder_params, "lr": 1e-7}, {"params": head_params, "lr": 1e-3}], weight_decay=0.01)
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.05), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"\nPOS unfrozen: {epochs} epochs, encoder LR=1e-7, head LR=1e-3\n")

best_acc = 0.856
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = forward_pos(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(pos_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in dev_loader:
            logits = forward_pos(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            mask = batch["labels"] != -100
            correct += (preds[mask] == batch["labels"][mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, POS Acc: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.pos_head.state_dict(), f"{DRIVE_OUT}/pos_head_best.pt")
        torch.save(model.deberta.state_dict(), f"{DRIVE_OUT}/encoder_pos_tuned.pt")
        print(f"  * New best Acc={acc:.4f}")
print(f"\nBest POS Acc: {best_acc:.4f}")

In [ ]:
# Cell 24: Quick SRL eval on the current model (after POS tuning)
import json, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import importlib, train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLDataset, collate_fn, evaluate, SRL_TAGS
from torch.utils.data import DataLoader

device = torch.device("cuda")

with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]

dev_loader = DataLoader(SRLDataset(gold_dev, tokenizer), batch_size=64, collate_fn=collate_fn)

# model is still in memory from Cell 23, with the POS-tuned encoder
# SRL uses the classifier head (not pos_head)
f1, report = evaluate(model, dev_loader, device, use_viterbi=True)
print(f"SRL F1 (Viterbi): {f1:.4f}")
print(report)

In [ ]:
# Cell 25: Save current model (POS 0.892 + SRL 0.696)
import torch, os
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
SAVE_DIR = f"{DRIVE_OUT}/pos_srl_v1_pos892_srl696"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save({
    "deberta": model.deberta.state_dict(),
    "pred_embedding": model.pred_embedding.state_dict(),
    "classifier": model.classifier.state_dict(),
    "pos_head": model.pos_head.state_dict(),
}, f"{SAVE_DIR}/model.pt")
model.deberta.config.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")
print(f"  POS Acc: 0.892 | SRL F1: 0.696 (Viterbi)")

In [ ]:
# Cell 26: POS — restart from gold_best_v1 (0.738 SRL), encoder 1e-8, head 1e-3
import json, torch, sys, os
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
os.makedirs(f"{DRIVE_OUT}/pos_srl_v2", exist_ok=True)

# Start fresh from gold_best_v1 (best SRL) + best frozen POS head
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best_v1_0738")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best_v1_0738/model.pt", weights_only=True)
model.load_state_dict(state)

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
model.pos_head = nn.Linear(model.deberta.config.hidden_size, len(pos_labels))
model.pos_head.load_state_dict(torch.load(f"{DRIVE_OUT}/pos_head_best.pt", weights_only=True))
model = model.to(device)

encoder_params, head_params = [], []
for n, p in model.named_parameters():
    p.requires_grad = True
    (head_params if "pos_head" in n else encoder_params).append(p)

def forward_pos(input_ids, attention_mask):
    return model.pos_head(model.deberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state)

class POSDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["pos_tags"][wid] if wid < len(ex["pos_tags"]) else "X", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
with open("data/prepared/kniv-deberta-cascade/ud_train.json") as f: ud_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f: ud_dev = json.load(f)
pos_map = {l: i for i, l in enumerate(pos_labels)}
train_loader = DataLoader(POSDataset(ud_train, tokenizer, pos_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(POSDataset(ud_dev, tokenizer, pos_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([{"params": encoder_params, "lr": 1e-8}, {"params": head_params, "lr": 1e-3}], weight_decay=0.01)
epochs = 10
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.05), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"POS: {epochs} epochs, encoder LR=1e-8, head LR=1e-3\n")

best_acc = 0.856
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = forward_pos(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(pos_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in dev_loader:
            logits = forward_pos(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            mask = batch["labels"] != -100
            correct += (preds[mask] == batch["labels"][mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, POS Acc: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        torch.save({"deberta": model.deberta.state_dict(), "pred_embedding": model.pred_embedding.state_dict(),
                     "classifier": model.classifier.state_dict(), "pos_head": model.pos_head.state_dict()},
                    f"{DRIVE_OUT}/pos_srl_v2/model.pt")
        tokenizer.save_pretrained(f"{DRIVE_OUT}/pos_srl_v2")
        model.deberta.config.save_pretrained(f"{DRIVE_OUT}/pos_srl_v2")
        print(f"  * New best Acc={acc:.4f}")

print(f"\nBest POS Acc: {best_acc:.4f}")

# Check SRL impact
import importlib, train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLDataset, collate_fn, evaluate
with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    srl_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]
srl_loader = DataLoader(SRLDataset(srl_dev, tokenizer), batch_size=64, collate_fn=collate_fn)
f1, _ = evaluate(model, srl_loader, device, use_viterbi=True)
print(f"SRL F1 (Viterbi): {f1:.4f}")

In [ ]:
# Cell 27: Save pos_srl_v2 (POS 0.871, SRL 0.732)
import torch, os
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
SAVE_DIR = f"{DRIVE_OUT}/pos_srl_v2_pos871_srl732"
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save({
    "deberta": model.deberta.state_dict(),
    "pred_embedding": model.pred_embedding.state_dict(),
    "classifier": model.classifier.state_dict(),
    "pos_head": model.pos_head.state_dict(),
}, f"{SAVE_DIR}/model.pt")
model.deberta.config.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")
print(f"  POS Acc: 0.871 | SRL F1: 0.732 (Viterbi)")

In [ ]:
# Cell 28: BiLSTM adapter between encoder and SRL head
# Load from pos_srl_v2 (POS 0.871 + SRL 0.732). Freeze everything.
# BiLSTM learns fast, SRL classifier very slow. Check both POS and SRL after.
import json, random, torch, sys, os, importlib
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS, TAG2ID
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load pos_srl_v2 (POS 0.871 + SRL 0.732)
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/pos_srl_v2_pos871_srl732")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()

# Load all weights including POS head
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
H = model.deberta.config.hidden_size
model.pos_head = nn.Linear(H, len(pos_labels))

state = torch.load(f"{DRIVE_OUT}/pos_srl_v2_pos871_srl732/model.pt", weights_only=True)
model.deberta.load_state_dict(state["deberta"])
model.pred_embedding.load_state_dict(state["pred_embedding"])
model.classifier.load_state_dict(state["classifier"])
model.pos_head.load_state_dict(state["pos_head"])
print("Loaded pos_srl_v2 (POS 0.871 + SRL 0.732)")

# Add BiLSTM adapter
lstm_hidden = 256
model.srl_lstm = nn.LSTM(H, lstm_hidden, num_layers=1, bidirectional=True, batch_first=True)
model.srl_proj = nn.Linear(lstm_hidden * 2, H)
model = model.to(device)

# Freeze encoder + pred_embedding + pos_head
lstm_params, classifier_params = [], []
for n, p in model.named_parameters():
    if "srl_lstm" in n or "srl_proj" in n:
        p.requires_grad = True
        lstm_params.append(p)
    elif "classifier" in n:
        p.requires_grad = True
        classifier_params.append(p)
    else:
        p.requires_grad = False

print(f"BiLSTM+proj: {sum(p.numel() for p in lstm_params):,} | Classifier: {sum(p.numel() for p in classifier_params):,}")

def forward_lstm_srl(input_ids, attention_mask, predicate_idx):
    B, S = input_ids.size()
    emb = model.deberta.embeddings(input_ids)
    indicator = torch.zeros(B, S, dtype=torch.long, device=input_ids.device)
    indicator[torch.arange(B, device=input_ids.device), predicate_idx] = 1
    emb = emb + model.pred_embedding(indicator)
    hidden = model.deberta.encoder(emb, attention_mask).last_hidden_state
    lstm_out, _ = model.srl_lstm(hidden)
    adapted = model.srl_proj(lstm_out) + hidden  # residual
    return model.classifier(adapted)

model.forward = forward_lstm_srl

# 20K gold
with open("data/prepared/kniv-deberta-cascade/srl_full_train.json") as f:
    gold_all = json.load(f)
gold_all = [ex for ex in gold_all if "predicate_idx" in ex and any(t not in ("O","V") for t in ex["srl_tags"])]
random.seed(42)
gold_train = random.sample(gold_all, min(20000, len(gold_all)))

with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    gold_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]

train_loader = DataLoader(SRLDataset(gold_train, tokenizer), batch_size=64, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(SRLDataset(gold_dev, tokenizer), batch_size=64, collate_fn=collate_fn)

optimizer = torch.optim.AdamW([
    {"params": lstm_params, "lr": 1e-3},
    {"params": classifier_params, "lr": 1e-5},
], weight_decay=0.01)

epochs = 10
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.05), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"\nBiLSTM experiment: {epochs} epochs, {len(train_loader)} batches/epoch\n")

best_f1 = 0.732
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = forward_lstm_srl(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["predicate_idx"].to(device))
        loss = loss_fn(logits.view(-1, len(SRL_TAGS)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()

    f1, _ = evaluate(model, dev_loader, device, use_viterbi=True)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, SRL F1 (VD): {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        os.makedirs(f"{DRIVE_OUT}/lstm_srl_best", exist_ok=True)
        torch.save({"srl_lstm": model.srl_lstm.state_dict(), "srl_proj": model.srl_proj.state_dict(),
                     "classifier": model.classifier.state_dict()}, f"{DRIVE_OUT}/lstm_srl_best/srl_layers.pt")
        print(f"  * New best F1={f1:.4f}")

# Check POS is intact
print(f"\nBest SRL F1: {best_f1:.4f}")
print("\nChecking POS...")
def forward_pos(input_ids, attention_mask):
    return model.pos_head(model.deberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state)

class POSDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["pos_tags"][wid] if wid < len(ex["pos_tags"]) else "X", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f: ud_dev = json.load(f)
pos_map = {l: i for i, l in enumerate(pos_labels)}
pos_loader = DataLoader(POSDataset(ud_dev, tokenizer, pos_map), batch_size=64,
    collate_fn=lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]})

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch in pos_loader:
        logits = forward_pos(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        preds = logits.argmax(dim=-1).cpu()
        mask = batch["labels"] != -100
        correct += (preds[mask] == batch["labels"][mask]).sum().item()
        total += mask.sum().item()
print(f"POS Acc: {correct/total:.4f} (should be ~0.871)")

In [ ]:
# Cell 29: Save BiLSTM model + CoNLL-2012 benchmark
import torch, os, json
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
SAVE_DIR = f"{DRIVE_OUT}/pos_srl_lstm_v1_pos871_srl742"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save({
    "deberta": model.deberta.state_dict(),
    "pred_embedding": model.pred_embedding.state_dict(),
    "classifier": model.classifier.state_dict(),
    "pos_head": model.pos_head.state_dict(),
    "srl_lstm": model.srl_lstm.state_dict(),
    "srl_proj": model.srl_proj.state_dict(),
}, f"{SAVE_DIR}/model.pt")
model.deberta.config.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")

# CoNLL-2012 benchmark
with open("data/prepared/kniv-deberta-cascade/conll2012_test.json") as f:
    conll_test = [ex for ex in json.load(f) if "predicate_idx" in ex]
print(f"\nCoNLL-2012 test: {len(conll_test)} examples")

from train_srl_encoder import SRLDataset, collate_fn, evaluate
from torch.utils.data import DataLoader
conll_loader = DataLoader(SRLDataset(conll_test, tokenizer), batch_size=64, collate_fn=collate_fn)
f1, report = evaluate(model, conll_loader, device, use_viterbi=True)
print(f"\n=== CoNLL-2012 Test — SRL F1: {f1:.4f} (SOTA: 87.67) ===")
print(report)

In [ ]:
# Cell 30: NER head — frozen encoder, cascade from POS
import json, torch, sys, importlib
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seq_f1, classification_report
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load pos_srl_lstm model
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/pos_srl_lstm_v1_pos871_srl742")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
ner_labels = vocabs["ner_labels"]
H = model.deberta.config.hidden_size

model.pos_head = nn.Linear(H, len(pos_labels))
# NER cascades from POS: input is [hidden, pos_probs]
model.ner_head = nn.Sequential(
    nn.LayerNorm(H + len(pos_labels)),
    nn.Linear(H + len(pos_labels), H),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(H, len(ner_labels)),
)

state = torch.load(f"{DRIVE_OUT}/pos_srl_lstm_v1_pos871_srl742/model.pt", weights_only=True)
model.deberta.load_state_dict(state["deberta"])
model.pred_embedding.load_state_dict(state["pred_embedding"])
model.classifier.load_state_dict(state["classifier"])
model.pos_head.load_state_dict(state["pos_head"])
model = model.to(device)
print("Loaded model. NER head randomly initialized.")

# Freeze everything except ner_head
for n, p in model.named_parameters():
    p.requires_grad = "ner_head" in n
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (ner_head only)")

def forward_ner(input_ids, attention_mask):
    hidden = model.deberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
    pos_logits = model.pos_head(hidden)
    pos_probs = torch.softmax(pos_logits, dim=-1).detach()
    ner_input = torch.cat([hidden.detach(), pos_probs], dim=-1)
    return model.ner_head(ner_input)

class NERDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["ner_tags"][wid] if wid < len(ex["ner_tags"]) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}

with open("data/prepared/kniv-deberta-cascade/ner_train.json") as f: ner_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ner_dev.json") as f: ner_dev = json.load(f)
ner_map = {l: i for i, l in enumerate(ner_labels)}
print(f"NER train: {len(ner_train):,}, dev: {len(ner_dev):,}, labels: {len(ner_labels)}")

train_loader = DataLoader(NERDataset(ner_train, tokenizer, ner_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(NERDataset(ner_dev[:2000], tokenizer, ner_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.1), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"\nNER training: {epochs} epochs, {len(train_loader)} batches/epoch\n")

best_f1 = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = forward_ner(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(ner_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        total_loss += loss.item()

    # Eval NER F1
    model.eval()
    all_gold, all_pred = [], []
    with torch.no_grad():
        for batch in dev_loader:
            logits = forward_ner(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            labels = batch["labels"]
            for j in range(labels.size(0)):
                gold_seq, pred_seq = [], []
                for k in range(labels.size(1)):
                    if labels[j, k] != -100:
                        gold_seq.append(ner_labels[labels[j, k]])
                        pred_seq.append(ner_labels[preds[j, k]] if preds[j, k] < len(ner_labels) else "O")
                all_gold.append(gold_seq)
                all_pred.append(pred_seq)
    f1 = seq_f1(all_gold, all_pred)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, NER F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.ner_head.state_dict(), f"{DRIVE_OUT}/ner_head_best.pt")
        print(f"  * New best F1={f1:.4f}")

print(f"\nBest NER F1: {best_f1:.4f}")

In [ ]:
# Save model with NER head
import torch, os
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
SAVE_DIR = f"{DRIVE_OUT}/pos_ner_srl_v1"
os.makedirs(SAVE_DIR, exist_ok=True)

# Load best NER head
model.ner_head.load_state_dict(torch.load(f"{DRIVE_OUT}/ner_head_best.pt", weights_only=True))

torch.save({
    "deberta": model.deberta.state_dict(),
    "pred_embedding": model.pred_embedding.state_dict(),
    "classifier": model.classifier.state_dict(),
    "pos_head": model.pos_head.state_dict(),
    "ner_head": model.ner_head.state_dict(),
}, f"{SAVE_DIR}/model.pt")
model.deberta.config.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")
print(f"  POS Acc: 0.871 | NER F1: 0.425 | SRL F1: 0.742 (VD)")

In [ ]:
# Cell 32: Save model with scalar mix POS
import torch, os
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
SAVE_DIR = f"{DRIVE_OUT}/pos966_ner425_srl742"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save({
    "deberta": model.deberta.state_dict(),
    "pred_embedding": model.pred_embedding.state_dict(),
    "classifier": model.classifier.state_dict(),
    "pos_head": model.pos_head.state_dict(),
    "pos_scalar_mix": model.pos_scalar_mix.state_dict(),
    "pos_head_mix": model.pos_head_mix.state_dict(),
    "ner_head": model.ner_head.state_dict(),
}, f"{SAVE_DIR}/model.pt")
model.deberta.config.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")
print(f"  POS Acc: 0.966 (scalar mix) | NER F1: 0.425 | SRL F1: 0.742 (VD)")

In [ ]:
# Cell 33: NER with Scalar Mix + POS cascade
import json, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seq_f1
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# model is still in memory from Cell 31 with all heads loaded

# Add NER scalar mix + new NER head
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
ner_labels = vocabs["ner_labels"]
pos_labels = vocabs["pos_labels"]
H = model.deberta.config.hidden_size
num_layers = model.deberta.config.num_hidden_layers + 1

class ScalarMix(nn.Module):
    def __init__(self, num_layers):
        super().__init__()
        self.weights = nn.Parameter(torch.zeros(num_layers))
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, all_layers):
        normed = torch.softmax(self.weights, dim=0)
        return self.scale * sum(w * layer for w, layer in zip(normed, all_layers))

model.ner_scalar_mix = ScalarMix(num_layers)
# NER head with POS cascade: [mixed_hidden + pos_probs] → MLP → NER labels
model.ner_head_mix = nn.Sequential(
    nn.LayerNorm(H + len(pos_labels)),
    nn.Linear(H + len(pos_labels), H),
    nn.GELU(), nn.Dropout(0.1),
    nn.Linear(H, len(ner_labels)),
)
model = model.to(device)

# Freeze everything except ner_scalar_mix + ner_head_mix
for n, p in model.named_parameters():
    p.requires_grad = "ner_scalar_mix" in n or "ner_head_mix" in n
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (ner_scalar_mix + ner_head_mix)")

def forward_ner_mix(input_ids, attention_mask):
    outputs = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    # NER reads from its own scalar mix
    ner_hidden = model.ner_scalar_mix(outputs.hidden_states)
    # POS reads from its own scalar mix for cascade
    pos_hidden = model.pos_scalar_mix(outputs.hidden_states)
    pos_probs = torch.softmax(model.pos_head_mix(pos_hidden), dim=-1).detach()
    ner_input = torch.cat([ner_hidden, pos_probs], dim=-1)
    return model.ner_head_mix(ner_input)

class NERDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["ner_tags"][wid] if wid < len(ex["ner_tags"]) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
with open("data/prepared/kniv-deberta-cascade/ner_train.json") as f: ner_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ner_dev.json") as f: ner_dev = json.load(f)
ner_map = {l: i for i, l in enumerate(ner_labels)}
train_loader = DataLoader(NERDataset(ner_train, tokenizer, ner_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(NERDataset(ner_dev[:2000], tokenizer, ner_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.1), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"\nNER Scalar Mix: {epochs} epochs, {len(train_loader)} batches/epoch\n")

best_f1 = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = forward_ner_mix(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(ner_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        total_loss += loss.item()

    model.eval()
    all_gold, all_pred = [], []
    with torch.no_grad():
        for batch in dev_loader:
            logits = forward_ner_mix(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            labels = batch["labels"]
            for j in range(labels.size(0)):
                gold_seq, pred_seq = [], []
                for k in range(labels.size(1)):
                    if labels[j, k] != -100:
                        gold_seq.append(ner_labels[labels[j, k]])
                        pred_seq.append(ner_labels[preds[j, k]] if preds[j, k] < len(ner_labels) else "O")
                all_gold.append(gold_seq)
                all_pred.append(pred_seq)
    f1 = seq_f1(all_gold, all_pred)
    w = torch.softmax(model.ner_scalar_mix.weights, dim=0)
    top3 = torch.topk(w, 3)
    layer_info = ", ".join(f"L{i}={v:.3f}" for i, v in zip(top3.indices.tolist(), top3.values.tolist()))
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, NER F1: {f1:.4f} | Top layers: {layer_info}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save({"ner_scalar_mix": model.ner_scalar_mix.state_dict(), "ner_head_mix": model.ner_head_mix.state_dict()},
                    f"{DRIVE_OUT}/ner_scalarmix_best.pt")
        print(f"  * New best F1={f1:.4f}")

print(f"\nBest NER F1: {best_f1:.4f}")
print(f"\nNER layer weights:")
w = torch.softmax(model.ner_scalar_mix.weights, dim=0)
for i, v in enumerate(w.tolist()):
    if v > 0.02: print(f"  Layer {i:2d}: {v:.4f} {'█' * int(v * 100)}")

In [ ]:
# Cell 34: NER error analysis
import json, torch
from seqeval.metrics import classification_report
from collections import Counter

model.eval()
all_gold, all_pred, all_words = [], [], []

with open("data/prepared/kniv-deberta-cascade/ner_dev.json") as f:
    ner_dev = json.load(f)

with torch.no_grad():
    for batch in dev_loader:
        logits = forward_ner_mix(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        preds = logits.argmax(dim=-1).cpu()
        labels = batch["labels"]
        for j in range(labels.size(0)):
            gold_seq, pred_seq = [], []
            for k in range(labels.size(1)):
                if labels[j, k] != -100:
                    gold_seq.append(ner_labels[labels[j, k]])
                    pred_seq.append(ner_labels[preds[j, k]] if preds[j, k] < len(ner_labels) else "O")
            all_gold.append(gold_seq)
            all_pred.append(pred_seq)

# Per-type report
print("=== Per-type NER Report ===")
print(classification_report(all_gold, all_pred, digits=3))

# Error analysis
invalid_bio = 0  # I-X without preceding B-X or I-X
type_mismatch = 0  # span detected but wrong type
missed_spans = 0  # gold span not detected at all
hallucinated = 0  # predicted span not in gold
boundary_errors = 0  # partial overlap

for gold_seq, pred_seq in zip(all_gold, all_pred):
    # Check invalid BIO in predictions
    prev_type = None
    for tag in pred_seq:
        if tag.startswith("I-"):
            cur_type = tag[2:]
            if prev_type != cur_type:
                invalid_bio += 1
        if tag.startswith("B-") or tag == "O":
            prev_type = tag[2:] if tag.startswith("B-") else None
        elif tag.startswith("I-"):
            prev_type = tag[2:]

# Count span-level errors
def extract_spans(tags):
    spans = []
    start, cur_type = None, None
    for i, tag in enumerate(tags):
        if tag.startswith("B-"):
            if start is not None:
                spans.append((start, i, cur_type))
            start, cur_type = i, tag[2:]
        elif tag.startswith("I-") and cur_type == tag[2:]:
            continue
        else:
            if start is not None:
                spans.append((start, i, cur_type))
            start, cur_type = None, None
    if start is not None:
        spans.append((start, len(tags), cur_type))
    return spans

for gold_seq, pred_seq in zip(all_gold, all_pred):
    gold_spans = set((s,e,t) for s,e,t in extract_spans(gold_seq))
    pred_spans = set((s,e,t) for s,e,t in extract_spans(pred_seq))

    for gs in gold_spans:
        if gs in pred_spans:
            continue
        # Check if there's a partial overlap
        partial = any(ps[0] <= gs[0] < ps[1] or gs[0] <= ps[0] < gs[1] for ps in pred_spans)
        if partial:
            boundary_errors += 1
        else:
            missed_spans += 1

    for ps in pred_spans:
        if ps not in gold_spans:
            partial = any(gs[0] <= ps[0] < gs[1] or ps[0] <= gs[0] < ps[1] for gs in gold_spans)
            if not partial:
                hallucinated += 1

print(f"\n=== Error Breakdown ===")
print(f"Invalid BIO sequences: {invalid_bio}")
print(f"Boundary errors (partial overlap): {boundary_errors}")
print(f"Missed spans (false negatives): {missed_spans}")
print(f"Hallucinated spans (false positives): {hallucinated}")

In [ ]:
# Cell 35: NER with CRF layer + Scalar Mix
# CRF learns transition probabilities, replaces both CrossEntropyLoss and Viterbi
!pip install -q pytorch-crf

import json, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seq_f1, classification_report
from torchcrf import CRF
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# model still in memory — reload best NER scalar mix weights
ner_sm_state = torch.load(f"{DRIVE_OUT}/ner_scalarmix_best.pt", weights_only=True)
model.ner_scalar_mix.load_state_dict(ner_sm_state["ner_scalar_mix"])
model.ner_head_mix.load_state_dict(ner_sm_state["ner_head_mix"])

# Add CRF layer on top of NER emissions
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
ner_labels = vocabs["ner_labels"]
pos_labels = vocabs["pos_labels"]

model.ner_crf = CRF(len(ner_labels), batch_first=True)
model = model.to(device)

# Freeze everything except ner_crf (and optionally ner_head_mix + ner_scalar_mix)
for n, p in model.named_parameters():
    p.requires_grad = "ner_crf" in n or "ner_scalar_mix" in n or "ner_head_mix" in n
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (ner_crf + ner_scalar_mix + ner_head_mix)")

def forward_ner_emissions(input_ids, attention_mask):
    outputs = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    ner_hidden = model.ner_scalar_mix(outputs.hidden_states)
    pos_hidden = model.pos_scalar_mix(outputs.hidden_states)
    pos_probs = torch.softmax(model.pos_head_mix(pos_hidden), dim=-1).detach()
    ner_input = torch.cat([ner_hidden, pos_probs], dim=-1)
    return model.ner_head_mix(ner_input)

class NERDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["ner_tags"][wid] if wid < len(ex["ner_tags"]) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
with open("data/prepared/kniv-deberta-cascade/ner_train.json") as f: ner_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ner_dev.json") as f: ner_dev = json.load(f)
ner_map = {l: i for i, l in enumerate(ner_labels)}
train_loader = DataLoader(NERDataset(ner_train, tokenizer, ner_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(NERDataset(ner_dev[:2000], tokenizer, ner_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([
    {"params": model.ner_crf.parameters(), "lr": 1e-2},  # CRF transitions learn fast
    {"params": list(model.ner_scalar_mix.parameters()) + list(model.ner_head_mix.parameters()), "lr": 5e-4},
], weight_decay=0.01)
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.1), epochs*len(train_loader))
print(f"\nNER + CRF: {epochs} epochs, {len(train_loader)} batches/epoch\n")

best_f1 = 0.687
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        emissions = forward_ner_emissions(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        labels = batch["labels"].to(device)
        mask = labels != -100

        # CRF needs contiguous labels — replace -100 with 0 (O tag) for masked positions
        crf_labels = labels.clone()
        crf_labels[~mask] = 0

        # CRF loss (negative log-likelihood)
        loss = -model.ner_crf(emissions, crf_labels, mask=mask, reduction="mean")

        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()

    # Eval with CRF Viterbi decode
    model.eval()
    all_gold, all_pred = [], []
    with torch.no_grad():
        for batch in dev_loader:
            emissions = forward_ner_emissions(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            labels = batch["labels"]
            mask = labels != -100

            # CRF decode
            decoded = model.ner_crf.decode(emissions, mask=mask.to(device))

            for j in range(labels.size(0)):
                gold_seq, pred_seq = [], []
                dec_idx = 0
                for k in range(labels.size(1)):
                    if labels[j, k] != -100:
                        gold_seq.append(ner_labels[labels[j, k]])
                        if dec_idx < len(decoded[j]):
                            pred_seq.append(ner_labels[decoded[j][dec_idx]] if decoded[j][dec_idx] < len(ner_labels) else "O")
                        else:
                            pred_seq.append("O")
                        dec_idx += 1
                all_gold.append(gold_seq)
                all_pred.append(pred_seq)
    f1 = seq_f1(all_gold, all_pred)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, NER F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save({"ner_crf": model.ner_crf.state_dict(), "ner_scalar_mix": model.ner_scalar_mix.state_dict(),
                     "ner_head_mix": model.ner_head_mix.state_dict()}, f"{DRIVE_OUT}/ner_crf_best.pt")
        print(f"  * New best F1={f1:.4f}")

print(f"\nBest NER F1: {best_f1:.4f}")

# Show what CRF learned
print("\nCRF transition matrix (top transitions):")
trans = model.ner_crf.transitions.data.cpu()
for i in range(len(ner_labels)):
    for j in range(len(ner_labels)):
        if abs(trans[i,j]) > 3.0:
            print(f"  {ner_labels[i]:>15} → {ner_labels[j]:<15}: {trans[i,j]:.2f}")

In [ ]:
# Cell 36a: Quick NER eval with Viterbi on existing 0.687 model (no retraining)
import torch, json
from train_srl_encoder import viterbi_decode

# Reload best NER weights (undo any CRF damage)
ner_sm_state = torch.load(f"{DRIVE_OUT}/ner_scalarmix_best.pt", weights_only=True)
model.ner_scalar_mix.load_state_dict(ner_sm_state["ner_scalar_mix"])
model.ner_head_mix.load_state_dict(ner_sm_state["ner_head_mix"])

ner_tag2id = {l: i for i, l in enumerate(ner_labels)}

model.eval()
all_gold_greedy, all_pred_greedy = [], []
all_gold_vd, all_pred_vd = [], []

with torch.no_grad():
    for batch in dev_loader:
        emissions = forward_ner_emissions(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        emissions_cpu = emissions.cpu()
        labels = batch["labels"]

        for j in range(labels.size(0)):
            valid_mask = labels[j] != -100
            valid_indices = valid_mask.nonzero(as_tuple=True)[0]
            if len(valid_indices) == 0:
                continue

            valid_logits = emissions_cpu[j, valid_indices]

            # Greedy
            greedy_ids = valid_logits.argmax(dim=-1).tolist()

            # Viterbi
            vd_ids = viterbi_decode(valid_logits, ner_tag2id)

            gold_seq_g, pred_seq_g = [], []
            gold_seq_v, pred_seq_v = [], []
            for k_idx, k in enumerate(valid_indices.tolist()):
                g = ner_labels[labels[j, k]]
                gold_seq_g.append(g)
                gold_seq_v.append(g)
                pred_seq_g.append(ner_labels[greedy_ids[k_idx]] if greedy_ids[k_idx] < len(ner_labels) else "O")
                pred_seq_v.append(ner_labels[vd_ids[k_idx]] if vd_ids[k_idx] < len(ner_labels) else "O")

            all_gold_greedy.append(gold_seq_g)
            all_pred_greedy.append(pred_seq_g)
            all_gold_vd.append(gold_seq_v)
            all_pred_vd.append(pred_seq_v)

from seqeval.metrics import f1_score as seq_f1
f1_greedy = seq_f1(all_gold_greedy, all_pred_greedy)
f1_vd = seq_f1(all_gold_vd, all_pred_vd)
print(f"NER Greedy:  {f1_greedy:.4f}")
print(f"NER Viterbi: {f1_vd:.4f}")
print(f"Improvement: +{f1_vd - f1_greedy:.4f}")

In [ ]:
# Cell 36b: NER with BIOES + CRF + Scalar Mix (from scratch)
import json, torch, sys, os
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seq_f1
from torchcrf import CRF
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# --- BIOES conversion ---
def bio_to_bioes(tags):
    """Convert BIO tags to BIOES."""
    result = []
    for i, tag in enumerate(tags):
        if tag == "O":
            result.append("O")
        elif tag.startswith("B-"):
            etype = tag[2:]
            # Check if next token is I-same_type
            if i + 1 < len(tags) and tags[i + 1] == f"I-{etype}":
                result.append(f"B-{etype}")  # start of multi-word
            else:
                result.append(f"S-{etype}")  # single-token entity
        elif tag.startswith("I-"):
            etype = tag[2:]
            # Check if next token is I-same_type
            if i + 1 < len(tags) and tags[i + 1] == f"I-{etype}":
                result.append(f"I-{etype}")  # inside
            else:
                result.append(f"E-{etype}")  # end of entity
        else:
            result.append(tag)
    return result

def bioes_to_spans(tags):
    """Convert BIOES tags to spans for seqeval (back to BIO)."""
    bio = []
    for tag in tags:
        if tag.startswith("S-"):
            bio.append(f"B-{tag[2:]}")
        elif tag.startswith("E-"):
            bio.append(f"I-{tag[2:]}")
        else:
            bio.append(tag)
    return bio

# Build BIOES label vocab from NER labels
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
ner_labels_bio = vocabs["ner_labels"]

# Extract entity types
entity_types = set()
for l in ner_labels_bio:
    if l.startswith("B-"):
        entity_types.add(l[2:])
entity_types = sorted(entity_types)

bioes_labels = ["O"]
for et in entity_types:
    bioes_labels.extend([f"B-{et}", f"I-{et}", f"E-{et}", f"S-{et}"])
bioes_map = {l: i for i, l in enumerate(bioes_labels)}
print(f"BIOES labels: {len(bioes_labels)} (was {len(ner_labels_bio)} in BIO)")

# Reload best NER scalar mix weights
ner_sm_state = torch.load(f"{DRIVE_OUT}/ner_scalarmix_best.pt", weights_only=True)
model.ner_scalar_mix.load_state_dict(ner_sm_state["ner_scalar_mix"])

# New BIOES head + CRF
H = model.deberta.config.hidden_size
pos_labels = vocabs["pos_labels"]
model.ner_bioes_head = nn.Sequential(
    nn.LayerNorm(H + len(pos_labels)),
    nn.Linear(H + len(pos_labels), H),
    nn.GELU(), nn.Dropout(0.1),
    nn.Linear(H, len(bioes_labels)),
)
model.ner_bioes_crf = CRF(len(bioes_labels), batch_first=True)
model = model.to(device)

# Freeze everything except bioes head + CRF + scalar mix
for n, p in model.named_parameters():
    p.requires_grad = "ner_bioes" in n or "ner_scalar_mix" in n
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,}")

def forward_bioes(input_ids, attention_mask):
    outputs = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    ner_hidden = model.ner_scalar_mix(outputs.hidden_states)
    pos_hidden = model.pos_scalar_mix(outputs.hidden_states)
    pos_probs = torch.softmax(model.pos_head_mix(pos_hidden), dim=-1).detach()
    return model.ner_bioes_head(torch.cat([ner_hidden, pos_probs], dim=-1))

class NERBIOESDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        bioes_tags = bio_to_bioes(ex["ner_tags"])
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(bioes_tags[wid] if wid < len(bioes_tags) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
train_loader = DataLoader(NERBIOESDataset(ner_train, tokenizer, bioes_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader_bioes = DataLoader(NERBIOESDataset(ner_dev[:2000], tokenizer, bioes_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([
    {"params": model.ner_bioes_crf.parameters(), "lr": 1e-2},
    {"params": list(model.ner_bioes_head.parameters()) + list(model.ner_scalar_mix.parameters()), "lr": 1e-3},
])
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.1), epochs*len(train_loader))
print(f"\nBIOES + CRF: {epochs} epochs, {len(train_loader)} batches/epoch\n")

best_f1 = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        emissions = forward_bioes(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        labels = batch["labels"].to(device)
        mask = labels != -100
        crf_labels = labels.clone(); crf_labels[~mask] = 0
        loss = -model.ner_bioes_crf(emissions, crf_labels, mask=mask, reduction="mean")
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()

    model.eval()
    all_gold, all_pred = [], []
    with torch.no_grad():
        for batch in dev_loader_bioes:
            emissions = forward_bioes(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            labels = batch["labels"]
            mask = labels != -100
            decoded = model.ner_bioes_crf.decode(emissions, mask=mask.to(device))
            for j in range(labels.size(0)):
                gold_seq, pred_seq = [], []
                dec_idx = 0
                for k in range(labels.size(1)):
                    if labels[j, k] != -100:
                        gold_seq.append(bioes_labels[labels[j, k]])
                        if dec_idx < len(decoded[j]):
                            pred_seq.append(bioes_labels[decoded[j][dec_idx]] if decoded[j][dec_idx] < len(bioes_labels) else "O")
                        else: pred_seq.append("O")
                        dec_idx += 1
                # Convert BIOES back to BIO for seqeval
                all_gold.append(bioes_to_spans(gold_seq))
                all_pred.append(bioes_to_spans(pred_seq))
    f1 = seq_f1(all_gold, all_pred)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, NER F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        os.makedirs(f"{DRIVE_OUT}/ner_bioes_crf_best", exist_ok=True)
        torch.save({"ner_bioes_head": model.ner_bioes_head.state_dict(),
                     "ner_bioes_crf": model.ner_bioes_crf.state_dict(),
                     "ner_scalar_mix": model.ner_scalar_mix.state_dict(),
                     "bioes_labels": bioes_labels},
                    f"{DRIVE_OUT}/ner_bioes_crf_best/ner_bioes.pt")
        print(f"  * New best F1={f1:.4f}")

print(f"\nBest NER F1: {best_f1:.4f}")

In [ ]:
# Cell 37: NER A/B experiments — test BIOES, head depth, all with VD
# Runs 4 configs sequentially, ~10 min each, reports comparison table
import json, torch, sys, os
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seq_f1
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# --- Shared setup ---
def bio_to_bioes(tags):
    result = []
    for i, tag in enumerate(tags):
        if tag == "O": result.append("O")
        elif tag.startswith("B-"):
            et = tag[2:]
            result.append(f"B-{et}" if i+1 < len(tags) and tags[i+1] == f"I-{et}" else f"S-{et}")
        elif tag.startswith("I-"):
            et = tag[2:]
            result.append(f"I-{et}" if i+1 < len(tags) and tags[i+1] == f"I-{et}" else f"E-{et}")
        else: result.append(tag)
    return result

def bioes_to_bio(tags):
    return [f"B-{t[2:]}" if t.startswith("S-") else f"I-{t[2:]}" if t.startswith("E-") else t for t in tags]

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
ner_labels_bio = vocabs["ner_labels"]
entity_types = sorted(set(l[2:] for l in ner_labels_bio if l.startswith("B-")))
bioes_labels = ["O"]
for et in entity_types:
    bioes_labels.extend([f"B-{et}", f"I-{et}", f"E-{et}", f"S-{et}"])

bio_map = {l: i for i, l in enumerate(ner_labels_bio)}
bioes_map = {l: i for i, l in enumerate(bioes_labels)}

sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRL_TAGS, viterbi_decode

with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_train.json") as f: ner_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_dev.json") as f: ner_dev = json.load(f)
print(f"SpanMarker train: {len(ner_train):,}, dev: {len(ner_dev):,}")

class ScalarMix(nn.Module):
    def __init__(self, n, favor_range=None):
        super().__init__()
        init = torch.zeros(n)
        if favor_range:
            for i in range(n):
                init[i] = 2.0 if favor_range[0] <= i <= favor_range[1] else -2.0
        self.weights = nn.Parameter(init)
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, layers):
        w = torch.softmax(self.weights, dim=0)
        return self.scale * sum(wi * li for wi, li in zip(w, layers))

class NERDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, use_bioes=False, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.use_bioes, self.max_length = examples, tokenizer, label_map, use_bioes, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        tags = bio_to_bioes(ex["ner_tags"]) if self.use_bioes else ex["ner_tags"]
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(tags[wid] if wid < len(tags) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}

def run_experiment(name, use_bioes, head_type, epochs=5):
    """Run one NER experiment. Returns best F1."""
    print(f"\n{'='*60}")
    print(f"Experiment: {name}")
    print(f"{'='*60}")

    labels = bioes_labels if use_bioes else ner_labels_bio
    lmap = bioes_map if use_bioes else bio_map
    num_labels = len(labels)

    # Fresh model each time
    tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/pos_srl_lstm_v1_pos871_srl742")
    mdl = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
    st = torch.load(f"{DRIVE_OUT}/pos_srl_lstm_v1_pos871_srl742/model.pt", weights_only=True)
    mdl.deberta.load_state_dict(st["deberta"])
    mdl.pred_embedding.load_state_dict(st["pred_embedding"])
    H = mdl.deberta.config.hidden_size
    num_layers = mdl.deberta.config.num_hidden_layers + 1

    # POS (frozen)
    mdl.pos_scalar_mix = ScalarMix(num_layers)
    mdl.pos_head_mix = nn.Linear(H, len(pos_labels))
    ps = torch.load(f"{DRIVE_OUT}/pos_scalarmix_best.pt", weights_only=True)
    mdl.pos_scalar_mix.load_state_dict(ps["pos_scalar_mix"])
    mdl.pos_head_mix.load_state_dict(ps["pos_head_mix"])

    # NER scalar mix + head
    mdl.ner_mix = ScalarMix(num_layers, favor_range=(5, 10))
    in_dim = H + len(pos_labels)

    if head_type == "linear":
        mdl.ner_head = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, num_labels))
    elif head_type == "mlp1":
        mdl.ner_head = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, H), nn.GELU(), nn.Dropout(0.1), nn.Linear(H, num_labels))
    elif head_type == "mlp2":
        mdl.ner_head = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, H), nn.GELU(), nn.Dropout(0.1), nn.Linear(H, H//2), nn.GELU(), nn.Dropout(0.1), nn.Linear(H//2, num_labels))

    mdl = mdl.to(device)
    for n, p in mdl.named_parameters():
        p.requires_grad = "ner_mix" in n or "ner_head" in n
    trainable = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
    print(f"Labels: {num_labels} ({'BIOES' if use_bioes else 'BIO'}), Head: {head_type}, Params: {trainable:,}")

    def fwd(input_ids, attention_mask):
        out = mdl.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        ner_h = mdl.ner_mix(out.hidden_states)
        pos_h = mdl.pos_scalar_mix(out.hidden_states)
        pos_p = torch.softmax(mdl.pos_head_mix(pos_h), dim=-1).detach()
        return mdl.ner_head(torch.cat([ner_h, pos_p], dim=-1))

    train_dl = DataLoader(NERDataset(ner_train, tokenizer, lmap, use_bioes), batch_size=64, shuffle=True, collate_fn=collate)
    dev_dl = DataLoader(NERDataset(ner_dev[:2000], tokenizer, lmap, use_bioes), batch_size=64, collate_fn=collate)

    opt = torch.optim.AdamW([p for p in mdl.parameters() if p.requires_grad], lr=1e-3)
    sched = get_linear_schedule_with_warmup(opt, int(epochs*len(train_dl)*0.1), epochs*len(train_dl))
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    best_f1 = 0.0
    for epoch in range(epochs):
        mdl.train()
        tl = 0.0
        for batch in tqdm(train_dl, desc=f"E{epoch+1}", leave=False):
            logits = fwd(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            loss = loss_fn(logits.view(-1, num_labels), batch["labels"].to(device).view(-1))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tl += loss.item()

        mdl.eval()
        all_g, all_p = [], []
        tag2id = lmap
        with torch.no_grad():
            for batch in dev_dl:
                logits = fwd(batch["input_ids"].to(device), batch["attention_mask"].to(device))
                logits_cpu = logits.cpu()
                labs = batch["labels"]
                for j in range(labs.size(0)):
                    valid_mask = labs[j] != -100
                    valid_idx = valid_mask.nonzero(as_tuple=True)[0]
                    if len(valid_idx) == 0: continue
                    vl = logits_cpu[j, valid_idx]
                    vd_ids = viterbi_decode(vl, tag2id)
                    gs, ps = [], []
                    for ki, k in enumerate(valid_idx.tolist()):
                        g = labels[labs[j, k]]
                        p = labels[vd_ids[ki]] if vd_ids[ki] < len(labels) else "O"
                        gs.append(g); ps.append(p)
                    if use_bioes:
                        gs = bioes_to_bio(gs); ps = bioes_to_bio(ps)
                    all_g.append(gs); all_p.append(ps)
        f1 = seq_f1(all_g, all_p)
        print(f"  Epoch {epoch+1}/{epochs} — Loss: {tl/len(train_dl):.4f}, NER F1: {f1:.4f}")
        if f1 > best_f1: best_f1 = f1

    print(f"  Best: {best_f1:.4f}")
    del mdl
    torch.cuda.empty_cache()
    return best_f1

# --- Run experiments ---
results = {}
results["BIO + Linear + VD"] = run_experiment("BIO + Linear + VD", use_bioes=False, head_type="linear")
results["BIO + MLP-1 + VD"] = run_experiment("BIO + MLP-1 + VD", use_bioes=False, head_type="mlp1")
results["BIOES + Linear + VD"] = run_experiment("BIOES + Linear + VD", use_bioes=True, head_type="linear")
results["BIOES + MLP-1 + VD"] = run_experiment("BIOES + MLP-1 + VD", use_bioes=True, head_type="mlp1")
results["BIOES + MLP-2 + VD"] = run_experiment("BIOES + MLP-2 + VD", use_bioes=True, head_type="mlp2")

print(f"\n{'='*60}")
print(f"RESULTS COMPARISON")
print(f"{'='*60}")
for name, f1 in sorted(results.items(), key=lambda x: -x[1]):
    bar = "█" * int(f1 * 50)
    print(f"  {f1:.4f}  {bar}  {name}")

In [ ]:
# Cell 38: NER BIO + MLP-1 + VD — 10 more epochs on SpanMarker data
# Continue from the winning config, more epochs since it was still climbing
import json, torch, sys
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seq_f1
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRL_TAGS, viterbi_decode

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
ner_labels = vocabs["ner_labels"]
bio_map = {l: i for i, l in enumerate(ner_labels)}

# Fresh model + heads
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/pos_srl_lstm_v1_pos871_srl742")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
st = torch.load(f"{DRIVE_OUT}/pos_srl_lstm_v1_pos871_srl742/model.pt", weights_only=True)
model.deberta.load_state_dict(st["deberta"])
model.pred_embedding.load_state_dict(st["pred_embedding"])
H = model.deberta.config.hidden_size
num_layers = model.deberta.config.num_hidden_layers + 1

class ScalarMix(nn.Module):
    def __init__(self, n, favor_range=None):
        super().__init__()
        init = torch.zeros(n)
        if favor_range:
            for i in range(n):
                init[i] = 2.0 if favor_range[0] <= i <= favor_range[1] else -2.0
        self.weights = nn.Parameter(init)
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, layers):
        w = torch.softmax(self.weights, dim=0)
        return self.scale * sum(wi * li for wi, li in zip(w, layers))

model.pos_scalar_mix = ScalarMix(num_layers)
model.pos_head_mix = nn.Linear(H, len(pos_labels))
ps = torch.load(f"{DRIVE_OUT}/pos_scalarmix_best.pt", weights_only=True)
model.pos_scalar_mix.load_state_dict(ps["pos_scalar_mix"])
model.pos_head_mix.load_state_dict(ps["pos_head_mix"])

model.ner_mix = ScalarMix(num_layers, favor_range=(5, 10))
in_dim = H + len(pos_labels)
model.ner_head = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, H), nn.GELU(), nn.Dropout(0.1), nn.Linear(H, len(ner_labels)))
model = model.to(device)

for n, p in model.named_parameters():
    p.requires_grad = "ner_mix" in n or "ner_head" in n

def fwd(input_ids, attention_mask):
    out = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    ner_h = model.ner_mix(out.hidden_states)
    pos_h = model.pos_scalar_mix(out.hidden_states)
    pos_p = torch.softmax(model.pos_head_mix(pos_h), dim=-1).detach()
    return model.ner_head(torch.cat([ner_h, pos_p], dim=-1))

class NERDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["ner_tags"][wid] if wid < len(ex["ner_tags"]) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_train.json") as f: ner_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_dev.json") as f: ner_dev = json.load(f)
train_loader = DataLoader(NERDataset(ner_train, tokenizer, bio_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(NERDataset(ner_dev[:2000], tokenizer, bio_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
epochs = 10
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.1), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"NER BIO+MLP-1+VD: {epochs} epochs, {len(train_loader)} batches/epoch\n")

import os
best_f1 = 0.0
for epoch in range(epochs):
    model.train()
    tl = 0.0
    for batch in tqdm(train_loader, desc=f"E{epoch+1}"):
        logits = fwd(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(ner_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        tl += loss.item()

    model.eval()
    all_g, all_p = [], []
    with torch.no_grad():
        for batch in dev_loader:
            logits = fwd(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            logits_cpu = logits.cpu()
            labs = batch["labels"]
            for j in range(labs.size(0)):
                vi = (labs[j] != -100).nonzero(as_tuple=True)[0]
                if len(vi) == 0: continue
                vl = logits_cpu[j, vi]
                vd = viterbi_decode(vl, bio_map)
                gs = [ner_labels[labs[j, k]] for k in vi.tolist()]
                ps = [ner_labels[vd[ki]] if vd[ki] < len(ner_labels) else "O" for ki in range(len(vi))]
                all_g.append(gs); all_p.append(ps)
    f1 = seq_f1(all_g, all_p)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {tl/len(train_loader):.4f}, NER F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        os.makedirs(f"{DRIVE_OUT}/ner_best_v2", exist_ok=True)
        torch.save({"ner_mix": model.ner_mix.state_dict(), "ner_head": model.ner_head.state_dict()},
                    f"{DRIVE_OUT}/ner_best_v2/best.pt")
        print(f"  * New best F1={f1:.4f}")

print(f"\nBest NER F1: {best_f1:.4f}")

In [ ]:
# Cell 39: Assemble the unified leader model
import torch, os, json, sys
import torch.nn as nn

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRL_TAGS
from transformers import AutoTokenizer

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
ner_labels = vocabs["ner_labels"]

model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
base = torch.load(f"{DRIVE_OUT}/pos_srl_lstm_v1_pos871_srl742/model.pt", weights_only=True)
model.deberta.load_state_dict(base["deberta"])
model.pred_embedding.load_state_dict(base["pred_embedding"])
model.classifier.load_state_dict(base["classifier"])
H = model.deberta.config.hidden_size
num_layers = model.deberta.config.num_hidden_layers + 1

class ScalarMix(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.weights = nn.Parameter(torch.zeros(n))
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, layers):
        w = torch.softmax(self.weights, dim=0)
        return self.scale * sum(wi * li for wi, li in zip(w, layers))

# POS
model.pos_scalar_mix = ScalarMix(num_layers)
model.pos_head = nn.Linear(H, len(pos_labels))
pos_state = torch.load(f"{DRIVE_OUT}/pos_scalarmix_best.pt", weights_only=True)
model.pos_scalar_mix.load_state_dict(pos_state["pos_scalar_mix"])
model.pos_head.load_state_dict(pos_state["pos_head_mix"])
print("POS: 0.966 Acc ✓")

# NER
model.ner_scalar_mix = ScalarMix(num_layers)
model.ner_head = nn.Sequential(
    nn.LayerNorm(H + len(pos_labels)),
    nn.Linear(H + len(pos_labels), H),
    nn.GELU(), nn.Dropout(0.1),
    nn.Linear(H, len(ner_labels)),
)
ner_state = torch.load(f"{DRIVE_OUT}/ner_best_v2/best.pt", weights_only=True)
model.ner_scalar_mix.load_state_dict(ner_state["ner_mix"])
model.ner_head.load_state_dict(ner_state["ner_head"])
print("NER: 0.846 F1 ✓")

# SRL BiLSTM
lstm_hidden = 256
model.srl_lstm = nn.LSTM(H, lstm_hidden, num_layers=1, bidirectional=True, batch_first=True)
model.srl_proj = nn.Linear(lstm_hidden * 2, H)
lstm_state = torch.load(f"{DRIVE_OUT}/lstm_srl_best/srl_layers.pt", weights_only=True)
model.srl_lstm.load_state_dict(lstm_state["srl_lstm"])
model.srl_proj.load_state_dict(lstm_state["srl_proj"])
model.classifier.load_state_dict(lstm_state["classifier"])
print("SRL: 0.742 F1 (VD) ✓")

model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/pos_srl_lstm_v1_pos871_srl742")

# Save unified leader
LEADER = f"{DRIVE_OUT}/leader_v1"
os.makedirs(LEADER, exist_ok=True)
torch.save({
    "deberta": model.deberta.state_dict(),
    "pred_embedding": model.pred_embedding.state_dict(),
    "pos_scalar_mix": model.pos_scalar_mix.state_dict(),
    "pos_head": model.pos_head.state_dict(),
    "ner_scalar_mix": model.ner_scalar_mix.state_dict(),
    "ner_head": model.ner_head.state_dict(),
    "classifier": model.classifier.state_dict(),
    "srl_lstm": model.srl_lstm.state_dict(),
    "srl_proj": model.srl_proj.state_dict(),
}, f"{LEADER}/model.pt")
model.deberta.config.save_pretrained(LEADER)
tokenizer.save_pretrained(LEADER)

meta = {
    "heads": {
        "pos": {"score": 0.966, "metric": "accuracy", "arch": "ScalarMix(0-8) + Linear", "decode": "argmax"},
        "ner": {"score": 0.846, "metric": "F1", "arch": "ScalarMix(5-10) + MLP-1", "decode": "viterbi"},
        "srl": {"score": 0.742, "metric": "F1", "arch": "BiLSTM + MLP + pred_embedding", "decode": "viterbi"},
    },
    "encoder": "DeBERTa-v3-large (SRL-tuned)",
    "version": "leader_v1",
}
with open(f"{LEADER}/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

size = os.path.getsize(f"{LEADER}/model.pt") / 1e6
print(f"\n{'='*50}")
print(f"LEADER v1 saved to {LEADER}")
print(f"{'='*50}")
print(f"  POS: 0.966 | NER: 0.846 | SRL: 0.742")
print(f"  DEP: not trained | CLS: not trained")
print(f"  Size: {size:.0f} MB")

In [ ]:
# Find the LSTM weights
import os
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
for root, dirs, files in os.walk(DRIVE_OUT):
    for f in files:
        if "lstm" in f.lower() or "srl_layers" in f.lower():
            print(os.path.join(root, f))

In [ ]:
# Cell 41: Verify all heads on the leader model
import json, torch, sys, importlib
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS, viterbi_decode
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seq_f1
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# model is still in memory from Cell 39 assembly

# === POS eval ===
print("=== POS ===")
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f:
    ud_dev = json.load(f)

class POSDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["pos_tags"][wid] if wid < len(ex["pos_tags"]) else "X", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

pos_map = {l: i for i, l in enumerate(pos_labels)}
pos_loader = DataLoader(POSDataset(ud_dev, tokenizer, pos_map), batch_size=64,
    collate_fn=lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]})

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch in pos_loader:
        out = model.deberta(batch["input_ids"].to(device), batch["attention_mask"].to(device), output_hidden_states=True)
        pos_h = model.pos_scalar_mix(out.hidden_states)
        logits = model.pos_head(pos_h)
        preds = logits.argmax(dim=-1).cpu()
        mask = batch["labels"] != -100
        correct += (preds[mask] == batch["labels"][mask]).sum().item()
        total += mask.sum().item()
print(f"POS Acc: {correct/total:.4f}")

# === NER eval ===
print("\n=== NER ===")
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_dev.json") as f:
    ner_dev = json.load(f)

class NERDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["ner_tags"][wid] if wid < len(ex["ner_tags"]) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

ner_map = {l: i for i, l in enumerate(ner_labels)}
ner_loader = DataLoader(NERDataset(ner_dev[:2000], tokenizer, ner_map), batch_size=64,
    collate_fn=lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]})

all_g, all_p = [], []
with torch.no_grad():
    for batch in ner_loader:
        out = model.deberta(batch["input_ids"].to(device), batch["attention_mask"].to(device), output_hidden_states=True)
        ner_h = model.ner_scalar_mix(out.hidden_states)
        pos_h = model.pos_scalar_mix(out.hidden_states)
        pos_probs = torch.softmax(model.pos_head(pos_h), dim=-1)
        logits = model.ner_head(torch.cat([ner_h, pos_probs], dim=-1)).cpu()
        labs = batch["labels"]
        for j in range(labs.size(0)):
            vi = (labs[j] != -100).nonzero(as_tuple=True)[0]
            if len(vi) == 0: continue
            vl = logits[j, vi]
            vd = viterbi_decode(vl, ner_map)
            all_g.append([ner_labels[labs[j, k]] for k in vi.tolist()])
            all_p.append([ner_labels[vd[ki]] if vd[ki] < len(ner_labels) else "O" for ki in range(len(vi))])
print(f"NER F1: {seq_f1(all_g, all_p):.4f}")

# === SRL eval ===
print("\n=== SRL ===")
with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    srl_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]

# Need to monkey-patch forward for BiLSTM path
def forward_srl(input_ids, attention_mask, predicate_idx):
    B, S = input_ids.size()
    emb = model.deberta.embeddings(input_ids)
    indicator = torch.zeros(B, S, dtype=torch.long, device=input_ids.device)
    indicator[torch.arange(B, device=input_ids.device), predicate_idx] = 1
    emb = emb + model.pred_embedding(indicator)
    hidden = model.deberta.encoder(emb, attention_mask).last_hidden_state
    lstm_out, _ = model.srl_lstm(hidden)
    adapted = model.srl_proj(lstm_out) + hidden
    return model.classifier(adapted)

model.forward = forward_srl
srl_loader = DataLoader(SRLDataset(srl_dev, tokenizer), batch_size=64, collate_fn=collate_fn)
f1_srl, _ = evaluate(model, srl_loader, device, use_viterbi=True)
print(f"SRL F1: {f1_srl:.4f} (Viterbi)")

print(f"\n{'='*50}")
print(f"LEADER v1 VERIFIED")
print(f"  POS: {correct/total:.4f} | NER: {seq_f1(all_g, all_p):.4f} | SRL: {f1_srl:.4f}")

In [ ]:
# Cell 42: Eval leader on standard benchmarks — CoNLL-2012 SRL + CoNLL-2003 NER
# POS doesn't have a CoNLL benchmark, we use UD EWT test instead
import json, torch, subprocess
from torch.utils.data import DataLoader
from seqeval.metrics import f1_score as seq_f1

# Pull latest (for conll2012_test.json)
subprocess.check_call(["git", "pull"])

# === POS on UD EWT TEST (not dev) ===
print("=== POS (UD EWT test) ===")
with open("data/prepared/kniv-deberta-cascade/ud_test.json") as f:
    ud_test = json.load(f)

pos_loader_test = DataLoader(POSDataset(ud_test, tokenizer, pos_map), batch_size=64,
    collate_fn=lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]})

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch in pos_loader_test:
        out = model.deberta(batch["input_ids"].to(device), batch["attention_mask"].to(device), output_hidden_states=True)
        logits = model.pos_head(model.pos_scalar_mix(out.hidden_states))
        preds = logits.argmax(dim=-1).cpu()
        mask = batch["labels"] != -100
        correct += (preds[mask] == batch["labels"][mask]).sum().item()
        total += mask.sum().item()
print(f"POS Acc: {correct/total:.4f} (on {len(ud_test)} test examples)")

# === NER on CoNLL-2003 test ===
print("\n=== NER (CoNLL-2003 test) ===")
# Check if we have it
conll03_path = "data/prepared/kniv-deberta-cascade/conll2003_test.json"
import os
if os.path.exists(conll03_path):
    with open(conll03_path) as f:
        conll03 = json.load(f)
    # Map CoNLL-2003 labels to our 18-type scheme
    # CoNLL-2003 uses: PER, ORG, LOC, MISC — we have PERSON, ORG, LOC, + more
    # This won't be a fair comparison, skip
    print("CoNLL-2003 not in our label scheme (4 types vs 18). Skipping.")
else:
    print("CoNLL-2003 test not available. Using SpanMarker test instead.")

# NER on SpanMarker dev (full, not subsampled)
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_dev.json") as f:
    ner_dev_full = json.load(f)
print(f"Evaluating on {len(ner_dev_full)} SpanMarker dev examples...")
ner_loader_full = DataLoader(NERDataset(ner_dev_full, tokenizer, ner_map), batch_size=64,
    collate_fn=lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]})

all_g, all_p = [], []
with torch.no_grad():
    for batch in tqdm(ner_loader_full, desc="NER eval"):
        out = model.deberta(batch["input_ids"].to(device), batch["attention_mask"].to(device), output_hidden_states=True)
        ner_h = model.ner_scalar_mix(out.hidden_states)
        pos_h = model.pos_scalar_mix(out.hidden_states)
        pos_probs = torch.softmax(model.pos_head(pos_h), dim=-1)
        logits = model.ner_head(torch.cat([ner_h, pos_probs], dim=-1)).cpu()
        labs = batch["labels"]
        for j in range(labs.size(0)):
            vi = (labs[j] != -100).nonzero(as_tuple=True)[0]
            if len(vi) == 0: continue
            vl = logits[j, vi]
            vd = viterbi_decode(vl, ner_map)
            all_g.append([ner_labels[labs[j, k]] for k in vi.tolist()])
            all_p.append([ner_labels[vd[ki]] if vd[ki] < len(ner_labels) else "O" for ki in range(len(vi))])
print(f"NER F1: {seq_f1(all_g, all_p):.4f} (full SpanMarker dev, {len(ner_dev_full)} examples)")

# === SRL on CoNLL-2012 test ===
print("\n=== SRL (CoNLL-2012 test) ===")
with open("data/prepared/kniv-deberta-cascade/conll2012_test.json") as f:
    conll_test = [ex for ex in json.load(f) if "predicate_idx" in ex]
print(f"Evaluating on {len(conll_test)} CoNLL-2012 test examples...")

srl_loader = DataLoader(SRLDataset(conll_test, tokenizer), batch_size=64, collate_fn=collate_fn)
f1_srl, report = evaluate(model, srl_loader, device, use_viterbi=True)
print(f"SRL F1: {f1_srl:.4f} (SOTA: 87.67)")

# Also SRL on PropBank EWT dev
with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    srl_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]
srl_dev_loader = DataLoader(SRLDataset(srl_dev, tokenizer), batch_size=64, collate_fn=collate_fn)
f1_srl_dev, _ = evaluate(model, srl_dev_loader, device, use_viterbi=True)
print(f"SRL F1: {f1_srl_dev:.4f} (PropBank EWT dev)")

print(f"\n{'='*50}")
print(f"LEADER v1 BENCHMARK RESULTS")
print(f"{'='*50}")
print(f"  POS:  {correct/total:.4f}  (UD EWT test)")
print(f"  NER:  {seq_f1(all_g, all_p):.4f}  (SpanMarker dev full)")
print(f"  SRL:  {f1_srl:.4f}  (CoNLL-2012 test, SOTA=0.877)")
print(f"  SRL:  {f1_srl_dev:.4f}  (PropBank EWT dev)")

In [ ]:
# Cell 43: SRL retraining on AllenNLP silver — tune top encoder layers + pred_embedding
import subprocess, json, torch, sys, os, importlib, random
os.chdir("/content/kniv-nlp-models")
subprocess.check_call(["git", "pull"])

sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
LEADER = f"{DRIVE_OUT}/leader_v1"

silver_path = "data/prepared/kniv-deberta-cascade/srl_allennlp_silver.json"
if not os.path.exists(silver_path):
    from huggingface_hub import hf_hub_download
    hf_hub_download("dragonscale-ai/kniv-corpus-en", "prepared/kniv-deberta-cascade/srl_allennlp_silver.json",
                    repo_type="dataset", local_dir="/content/kniv-data")
    if not os.path.exists(silver_path):
        os.symlink("/content/kniv-data/prepared/kniv-deberta-cascade/srl_allennlp_silver.json", silver_path)

tokenizer = AutoTokenizer.from_pretrained(LEADER)
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{LEADER}/model.pt", weights_only=True)
model.deberta.load_state_dict(state["deberta"])
model.pred_embedding.load_state_dict(state["pred_embedding"])
model.classifier.load_state_dict(state["classifier"])
H = model.deberta.config.hidden_size

model.srl_lstm = nn.LSTM(H, 256, num_layers=1, bidirectional=True, batch_first=True)
model.srl_proj = nn.Linear(512, H)
model.srl_lstm.load_state_dict(state["srl_lstm"])
model.srl_proj.load_state_dict(state["srl_proj"])
model = model.to(device)

# Freeze all, then unfreeze SRL + top layers + pred_embedding
for p in model.parameters():
    p.requires_grad = False

srl_params, encoder_top_params = [], []
seen_ids = set()

for n, p in model.named_parameters():
    if "srl_lstm" in n or "srl_proj" in n or "classifier" in n:
        p.requires_grad = True
        srl_params.append(p)
        seen_ids.add(id(p))
    elif "pred_embedding" in n:
        p.requires_grad = True
        encoder_top_params.append(p)
        seen_ids.add(id(p))

for n, p in model.named_parameters():
    if id(p) in seen_ids:
        continue
    for layer_idx in [21, 22, 23]:
        if f"deberta.encoder.layer.{layer_idx}." in n:
            p.requires_grad = True
            encoder_top_params.append(p)
            seen_ids.add(id(p))
            break

print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"  SRL head: {sum(p.numel() for p in srl_params):,}")
print(f"  Top layers + pred_emb: {sum(p.numel() for p in encoder_top_params):,}")

def forward_srl(input_ids, attention_mask, predicate_idx):
    B, S = input_ids.size()
    emb = model.deberta.embeddings(input_ids)
    indicator = torch.zeros(B, S, dtype=torch.long, device=input_ids.device)
    indicator[torch.arange(B, device=input_ids.device), predicate_idx] = 1
    emb = emb + model.pred_embedding(indicator)
    hidden = model.deberta.encoder(emb, attention_mask).last_hidden_state
    lstm_out, _ = model.srl_lstm(hidden)
    adapted = model.srl_proj(lstm_out) + hidden
    return model.classifier(adapted)

model.forward = forward_srl

random.seed(42)
with open(silver_path) as f:
    silver = json.load(f)
silver_train = random.sample(silver, min(200000, len(silver)))
print(f"AllenNLP silver: {len(silver_train):,}")

with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f:
    srl_dev = [ex for ex in json.load(f) if "predicate_idx" in ex]

train_loader = DataLoader(SRLDataset(silver_train, tokenizer), batch_size=64, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(SRLDataset(srl_dev, tokenizer), batch_size=64, collate_fn=collate_fn)

optimizer = torch.optim.AdamW([
    {"params": srl_params, "lr": 1e-3},
    {"params": encoder_top_params, "lr": 1e-7},
], weight_decay=0.01)
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.1), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"\nSRL + top layers: {epochs} epochs, {len(train_loader)} batches/epoch")
print(f"SRL LR=1e-3, Top layers LR=1e-7\n")

best_f1 = 0.742
for epoch in range(epochs):
    model.train()
    tl = 0.0
    for batch in tqdm(train_loader, desc=f"E{epoch+1}"):
        logits = forward_srl(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["predicate_idx"].to(device))
        loss = loss_fn(logits.view(-1, len(SRL_TAGS)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        tl += loss.item()

    f1, _ = evaluate(model, dev_loader, device, use_viterbi=True)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {tl/len(train_loader):.4f}, SRL F1 (VD): {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save({
            "deberta": model.deberta.state_dict(),
            "pred_embedding": model.pred_embedding.state_dict(),
            "pos_scalar_mix": state["pos_scalar_mix"],
            "pos_head": state["pos_head"],
            "ner_scalar_mix": state["ner_scalar_mix"],
            "ner_head": state["ner_head"],
            "classifier": model.classifier.state_dict(),
            "srl_lstm": model.srl_lstm.state_dict(),
            "srl_proj": model.srl_proj.state_dict(),
        }, f"{LEADER}/model.pt")
        with open(f"{LEADER}/metadata.json") as f:
            meta = json.load(f)
        meta["heads"]["srl"]["score"] = round(best_f1, 4)
        meta["version"] = "leader_v1.1"
        with open(f"{LEADER}/metadata.json", "w") as f:
            json.dump(meta, f, indent=2)
        print(f"  * New best F1={f1:.4f} — leader updated")

print(f"\nBest SRL F1: {best_f1:.4f}")

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable: 42,034,218
  SRL head: 4,243,498
  Top layers + pred_emb: 37,790,720
AllenNLP silver: 200,000

SRL + top layers: 5 epochs, 3125 batches/epoch
SRL LR=1e-3, Top layers LR=1e-7



E1: 100%|██████████| 3125/3125 [17:26<00:00,  2.99it/s]


Epoch 1/5 — Loss: 0.3898, SRL F1 (VD): 0.7786
  * New best F1=0.7786 — leader updated


...

Epoch 3/5 — Loss: 0.3001, SRL F1 (VD): 0.7867


E4: 100%|██████████| 3125/3125 [17:25<00:00,  2.99it/s]


Epoch 4/5 — Loss: 0.2795, SRL F1 (VD): 0.7953


E5: 100%|██████████| 3125/3125 [17:24<00:00,  2.99it/s]


Epoch 5/5 — Loss: 0.2609, SRL F1 (VD): 0.8006
  * New best F1=0.8006 — leader updated

Best SRL F1: 0.8006


In [56]:
# Cell 44: SRL 5 more epochs — continue from leader (0.801)
# Model still in memory, just reset optimizer with lower LR

optimizer = torch.optim.AdamW([
    {"params": srl_params, "lr": 5e-4},
    {"params": encoder_top_params, "lr": 5e-8},
], weight_decay=0.01)
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.05), epochs*len(train_loader))
print(f"SRL continued: {epochs} epochs, SRL LR=5e-4, Top layers LR=5e-8\n")

best_f1 = 0.801
for epoch in range(epochs):
    model.train()
    tl = 0.0
    for batch in tqdm(train_loader, desc=f"E{epoch+1}"):
        logits = forward_srl(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["predicate_idx"].to(device))
        loss = loss_fn(logits.view(-1, len(SRL_TAGS)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        tl += loss.item()

    f1, _ = evaluate(model, dev_loader, device, use_viterbi=True)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {tl/len(train_loader):.4f}, SRL F1 (VD): {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save({
            "deberta": model.deberta.state_dict(),
            "pred_embedding": model.pred_embedding.state_dict(),
            "pos_scalar_mix": state["pos_scalar_mix"],
            "pos_head": state["pos_head"],
            "ner_scalar_mix": state["ner_scalar_mix"],
            "ner_head": state["ner_head"],
            "classifier": model.classifier.state_dict(),
            "srl_lstm": model.srl_lstm.state_dict(),
            "srl_proj": model.srl_proj.state_dict(),
        }, f"{LEADER}/model.pt")
        with open(f"{LEADER}/metadata.json") as f:
            meta = json.load(f)
        meta["heads"]["srl"]["score"] = round(best_f1, 4)
        meta["version"] = "leader_v1.2"
        with open(f"{LEADER}/metadata.json", "w") as f:
            json.dump(meta, f, indent=2)
        print(f"  * New best F1={f1:.4f} — leader updated")

print(f"\nBest SRL F1: {best_f1:.4f}")

SRL continued: 5 epochs, SRL LR=5e-4, Top layers LR=5e-8



E1: 100%|██████████| 3125/3125 [17:32<00:00,  2.97it/s]


Epoch 1/5 — Loss: 0.2714, SRL F1 (VD): 0.8021
  * New best F1=0.8021 — leader updated


E2: 100%|██████████| 3125/3125 [17:26<00:00,  2.99it/s]


Epoch 2/5 — Loss: 0.2666, SRL F1 (VD): 0.7994


...

Epoch 3/5 — Loss: 0.2556, SRL F1 (VD): 0.7966


E4: 100%|██████████| 3125/3125 [17:26<00:00,  2.99it/s]


Epoch 4/5 — Loss: 0.2448, SRL F1 (VD): 0.7973


E5:   2%|▏         | 72/3125 [00:24<17:16,  2.95it/s]


KeyboardInterrupt: 

In [57]:
# Cell 45: Train DEP + CLS sequentially, versioned leader saves
import subprocess, json, torch, sys, os, importlib, shutil
os.chdir("/content/kniv-nlp-models")
subprocess.check_call(["git", "pull"])

sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRL_TAGS, viterbi_decode
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm
from seqeval.metrics import f1_score as seq_f1
from sklearn.metrics import f1_score as clf_f1

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
LEADER = f"{DRIVE_OUT}/leader_v1"

# --- Helper: save versioned leader ---
def save_leader(model, state, version, head_scores, tokenizer):
    """Save leader and a versioned backup."""
    # Save to leader
    save_dict = {
        "deberta": model.deberta.state_dict(),
        "pred_embedding": model.pred_embedding.state_dict(),
        "pos_scalar_mix": state.get("pos_scalar_mix", {}),
        "pos_head": state.get("pos_head", {}),
        "ner_scalar_mix": state.get("ner_scalar_mix", {}),
        "ner_head": state.get("ner_head", {}),
        "classifier": state.get("classifier", model.classifier.state_dict()),
        "srl_lstm": state.get("srl_lstm", {}),
        "srl_proj": state.get("srl_proj", {}),
    }
    # Add new heads if they exist
    for key in ["dep_scalar_mix", "dep_head", "cls_scalar_mix", "cls_head"]:
        if hasattr(model, key):
            save_dict[key] = getattr(model, key).state_dict()
        elif key in state:
            save_dict[key] = state[key]

    torch.save(save_dict, f"{LEADER}/model.pt")

    # Update metadata
    meta = {"heads": head_scores, "encoder": "DeBERTa-v3-large (SRL-tuned)", "version": version}
    with open(f"{LEADER}/metadata.json", "w") as f:
        json.dump(meta, f, indent=2)

    # Versioned backup
    backup = f"{DRIVE_OUT}/leader_{version}"
    os.makedirs(backup, exist_ok=True)
    shutil.copy2(f"{LEADER}/model.pt", f"{backup}/model.pt")
    shutil.copy2(f"{LEADER}/metadata.json", f"{backup}/metadata.json")
    print(f"  Leader saved: {version} → {backup}")

# --- Load leader ---
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
ner_labels = vocabs["ner_labels"]
dep_labels = vocabs["dep_labels"]
cls_labels = vocabs["cls_labels"]

tokenizer = AutoTokenizer.from_pretrained(LEADER)
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{LEADER}/model.pt", weights_only=True)
model.deberta.load_state_dict(state["deberta"])
model.pred_embedding.load_state_dict(state["pred_embedding"])
model.classifier.load_state_dict(state["classifier"])
H = model.deberta.config.hidden_size
num_layers = model.deberta.config.num_hidden_layers + 1

class ScalarMix(nn.Module):
    def __init__(self, n, favor_range=None):
        super().__init__()
        init = torch.zeros(n)
        if favor_range:
            for i in range(n):
                init[i] = 2.0 if favor_range[0] <= i <= favor_range[1] else -2.0
        self.weights = nn.Parameter(init)
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, layers):
        w = torch.softmax(self.weights, dim=0)
        return self.scale * sum(wi * li for wi, li in zip(w, layers))

# Load existing heads
model.pos_scalar_mix = ScalarMix(num_layers)
model.pos_head = nn.Linear(H, len(pos_labels))
model.pos_scalar_mix.load_state_dict(state["pos_scalar_mix"])
model.pos_head.load_state_dict(state["pos_head"])

model.ner_scalar_mix = ScalarMix(num_layers)
model.ner_head = nn.Sequential(nn.LayerNorm(H + len(pos_labels)), nn.Linear(H + len(pos_labels), H), nn.GELU(), nn.Dropout(0.1), nn.Linear(H, len(ner_labels)))
model.ner_scalar_mix.load_state_dict(state["ner_scalar_mix"])
model.ner_head.load_state_dict(state["ner_head"])

# BiLSTM for SRL
model.srl_lstm = nn.LSTM(H, 256, num_layers=1, bidirectional=True, batch_first=True)
model.srl_proj = nn.Linear(512, H)
model.srl_lstm.load_state_dict(state["srl_lstm"])
model.srl_proj.load_state_dict(state["srl_proj"])

print("Leader loaded: POS=0.966, NER=0.846, SRL=0.802")

# Current scores
head_scores = {
    "pos": {"score": 0.966, "metric": "accuracy"},
    "ner": {"score": 0.846, "metric": "F1"},
    "srl": {"score": 0.802, "metric": "F1"},
}

# --- Shared dataset/collate ---
class TokenDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, label_key, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.label_key, self.max_length = examples, tokenizer, label_map, label_key, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        tags = ex[self.label_key]
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(tags[wid] if wid < len(tags) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

class CLSDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        text = ex.get("text", " ".join(ex.get("words", [])))
        prev_text = ex.get("prev_text")
        if prev_text:
            enc = self.tokenizer(prev_text, text, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        else:
            enc = self.tokenizer(text, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(self.label_map[ex["cls_label"]], dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}

# ============================================================
# PART 1: DEP
# ============================================================
print(f"\n{'='*60}")
print("TRAINING DEP")
print(f"{'='*60}")

# Add DEP head: ScalarMix(12-18) + MLP with POS+NER cascade
model.dep_scalar_mix = ScalarMix(num_layers, favor_range=(12, 18))
in_dim = H + len(pos_labels) + len(ner_labels)
model.dep_head = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, H), nn.GELU(), nn.Dropout(0.1), nn.Linear(H, len(dep_labels)))
model = model.to(device)

for n, p in model.named_parameters():
    p.requires_grad = "dep_scalar_mix" in n or "dep_head" in n
print(f"DEP trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def fwd_dep(input_ids, attention_mask):
    out = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    dep_h = model.dep_scalar_mix(out.hidden_states)
    pos_h = model.pos_scalar_mix(out.hidden_states)
    pos_probs = torch.softmax(model.pos_head(pos_h), dim=-1).detach()
    ner_h = model.ner_scalar_mix(out.hidden_states)
    ner_probs = torch.softmax(model.ner_head(torch.cat([ner_h, pos_probs], dim=-1)), dim=-1).detach()
    return model.dep_head(torch.cat([dep_h, pos_probs, ner_probs], dim=-1))

with open("data/prepared/kniv-deberta-cascade/ud_train.json") as f: ud_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f: ud_dev = json.load(f)
dep_map = {l: i for i, l in enumerate(dep_labels)}

dep_train_loader = DataLoader(TokenDataset(ud_train, tokenizer, dep_map, "dep_labels"), batch_size=64, shuffle=True, collate_fn=collate)
dep_dev_loader = DataLoader(TokenDataset(ud_dev, tokenizer, dep_map, "dep_labels"), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
epochs = 10
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(dep_train_loader)*0.1), epochs*len(dep_train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"DEP: {epochs} epochs, {len(dep_train_loader)} batches/epoch\n")

best_dep = 0.0
for epoch in range(epochs):
    model.train()
    tl = 0.0
    for batch in tqdm(dep_train_loader, desc=f"DEP E{epoch+1}"):
        logits = fwd_dep(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(dep_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        tl += loss.item()

    # Eval: accuracy (UAS proxy for dep2label)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in dep_dev_loader:
            logits = fwd_dep(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            mask = batch["labels"] != -100
            correct += (preds[mask] == batch["labels"][mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total
    print(f"DEP E{epoch+1}/{epochs} — Loss: {tl/len(dep_train_loader):.4f}, Acc: {acc:.4f}")
    if acc > best_dep:
        best_dep = acc
        state["dep_scalar_mix"] = model.dep_scalar_mix.state_dict()
        state["dep_head"] = model.dep_head.state_dict()
        head_scores["dep"] = {"score": round(best_dep, 4), "metric": "accuracy"}
        save_leader(model, state, f"v2.0_dep{round(best_dep*1000)}", head_scores, tokenizer)

print(f"\nBest DEP Acc: {best_dep:.4f}")

# ============================================================
# PART 2: CLS
# ============================================================
print(f"\n{'='*60}")
print("TRAINING CLS")
print(f"{'='*60}")

# Add CLS head: ScalarMix + mean pooling + MLP
model.cls_scalar_mix = ScalarMix(num_layers, favor_range=(18, 23))
model.cls_head = nn.Sequential(nn.LayerNorm(H), nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(0.1), nn.Linear(H // 2, len(cls_labels)))
model = model.to(device)

for n, p in model.named_parameters():
    p.requires_grad = "cls_scalar_mix" in n or "cls_head" in n
print(f"CLS trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def fwd_cls(input_ids, attention_mask):
    out = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    cls_h = model.cls_scalar_mix(out.hidden_states)
    # Mean pooling over non-pad tokens
    mask = attention_mask.unsqueeze(-1).float()
    pooled = (cls_h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return model.cls_head(pooled)

with open("data/prepared/kniv-deberta-cascade/cls_swda_mrda_train.json") as f: cls_train = json.load(f)
cls_map = {l: i for i, l in enumerate(cls_labels)}
# Use a small dev set from the training data (last 5K)
import random
random.seed(42)
random.shuffle(cls_train)
cls_dev_data = cls_train[:5000]
cls_train_data = cls_train[5000:]
print(f"CLS train: {len(cls_train_data):,}, dev: {len(cls_dev_data):,}, labels: {len(cls_labels)}")

cls_train_loader = DataLoader(CLSDataset(cls_train_data, tokenizer, cls_map), batch_size=64, shuffle=True, collate_fn=collate)
cls_dev_loader = DataLoader(CLSDataset(cls_dev_data, tokenizer, cls_map), batch_size=64, collate_fn=collate)

# Weighted loss for class imbalance
from collections import Counter
dist = Counter(ex["cls_label"] for ex in cls_train_data)
total_ex = sum(dist.values())
weights = torch.tensor([total_ex / (len(cls_labels) * dist[l]) for l in cls_labels], dtype=torch.float, device=device)
cls_loss_fn = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(cls_train_loader)*0.1), epochs*len(cls_train_loader))
print(f"CLS: {epochs} epochs, {len(cls_train_loader)} batches/epoch\n")

best_cls = 0.0
for epoch in range(epochs):
    model.train()
    tl = 0.0
    for batch in tqdm(cls_train_loader, desc=f"CLS E{epoch+1}"):
        logits = fwd_cls(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = cls_loss_fn(logits, batch["labels"].to(device))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        tl += loss.item()

    model.eval()
    all_gold, all_pred = [], []
    with torch.no_grad():
        for batch in cls_dev_loader:
            logits = fwd_cls(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu().tolist()
            golds = batch["labels"].tolist()
            all_gold.extend(golds); all_pred.extend(preds)
    macro_f1 = clf_f1(all_gold, all_pred, average="macro")
    print(f"CLS E{epoch+1}/{epochs} — Loss: {tl/len(cls_train_loader):.4f}, Macro F1: {macro_f1:.4f}")
    if macro_f1 > best_cls:
        best_cls = macro_f1
        state["cls_scalar_mix"] = model.cls_scalar_mix.state_dict()
        state["cls_head"] = model.cls_head.state_dict()
        head_scores["cls"] = {"score": round(best_cls, 4), "metric": "macro_f1"}
        save_leader(model, state, f"v2.1_cls{round(best_cls*1000)}", head_scores, tokenizer)

print(f"\nBest CLS Macro F1: {best_cls:.4f}")

# Final summary
print(f"\n{'='*60}")
print("LEADER UPDATED")
print(f"{'='*60}")
for head, info in head_scores.items():
    print(f"  {head.upper():>4}: {info['score']:.4f} ({info['metric']})")

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Leader loaded: POS=0.966, NER=0.846, SRL=0.802

TRAINING DEP
DEP trainable: 2,583,078
DEP: 10 epochs, 196 batches/epoch



DEP E1: 100%|██████████| 196/196 [00:34<00:00,  5.70it/s]


DEP E1/10 — Loss: 2.9801, Acc: 0.6406
  Leader saved: v2.0_dep641 → /content/drive/MyDrive/kniv-models/srl_encoder/leader_v2.0_dep641


...

CLS E3/5 — Loss: 0.7872, Macro F1: 0.5159
  Leader saved: v2.1_cls516 → /content/drive/MyDrive/kniv-models/srl_encoder/leader_v2.1_cls516


CLS E4: 100%|██████████| 4391/4391 [12:12<00:00,  5.99it/s]


CLS E4/5 — Loss: 0.7089, Macro F1: 0.5216
  Leader saved: v2.1_cls522 → /content/drive/MyDrive/kniv-models/srl_encoder/leader_v2.1_cls522


CLS E5: 100%|██████████| 4391/4391 [12:12<00:00,  6.00it/s]


CLS E5/5 — Loss: 0.6385, Macro F1: 0.5250
  Leader saved: v2.1_cls525 → /content/drive/MyDrive/kniv-models/srl_encoder/leader_v2.1_cls525

Best CLS Macro F1: 0.5250

LEADER UPDATED
   POS: 0.9660 (accuracy)
   NER: 0.8460 (F1)
   SRL: 0.8020 (F1)
   DEP: 0.7427 (accuracy)
   CLS: 0.5250 (macro_f1)


In [2]:
# Cell 46: DEP — 20 more epochs (full reload since runtime lost context)
import subprocess, json, torch, sys, os, importlib
os.chdir("/content/kniv-nlp-models")
subprocess.check_call(["git", "pull"])

sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRL_TAGS, viterbi_decode
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm
import shutil

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
LEADER = f"{DRIVE_OUT}/leader_v1"

def save_leader(state_dict, version, head_scores):
    torch.save(state_dict, f"{LEADER}/model.pt")
    meta = {"heads": head_scores, "encoder": "DeBERTa-v3-large (SRL-tuned)", "version": version}
    with open(f"{LEADER}/metadata.json", "w") as f:
        json.dump(meta, f, indent=2)
    backup = f"{DRIVE_OUT}/leader_{version}"
    os.makedirs(backup, exist_ok=True)
    shutil.copy2(f"{LEADER}/model.pt", f"{backup}/model.pt")
    shutil.copy2(f"{LEADER}/metadata.json", f"{backup}/metadata.json")
    print(f"  Leader saved: {version}")

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels, ner_labels, dep_labels, cls_labels = vocabs["pos_labels"], vocabs["ner_labels"], vocabs["dep_labels"], vocabs["cls_labels"]

tokenizer = AutoTokenizer.from_pretrained(LEADER)
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{LEADER}/model.pt", weights_only=True)
model.deberta.load_state_dict(state["deberta"])
model.pred_embedding.load_state_dict(state["pred_embedding"])
model.classifier.load_state_dict(state["classifier"])
H = model.deberta.config.hidden_size
num_layers = model.deberta.config.num_hidden_layers + 1

class ScalarMix(nn.Module):
    def __init__(self, n, favor_range=None):
        super().__init__()
        init = torch.zeros(n)
        if favor_range:
            for i in range(n):
                init[i] = 2.0 if favor_range[0] <= i <= favor_range[1] else -2.0
        self.weights = nn.Parameter(init)
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, layers):
        w = torch.softmax(self.weights, dim=0)
        return self.scale * sum(wi * li for wi, li in zip(w, layers))

# Load all existing heads
model.pos_scalar_mix = ScalarMix(num_layers)
model.pos_head = nn.Linear(H, len(pos_labels))
model.pos_scalar_mix.load_state_dict(state["pos_scalar_mix"])
model.pos_head.load_state_dict(state["pos_head"])

model.ner_scalar_mix = ScalarMix(num_layers)
model.ner_head = nn.Sequential(nn.LayerNorm(H + len(pos_labels)), nn.Linear(H + len(pos_labels), H), nn.GELU(), nn.Dropout(0.1), nn.Linear(H, len(ner_labels)))
model.ner_scalar_mix.load_state_dict(state["ner_scalar_mix"])
model.ner_head.load_state_dict(state["ner_head"])

model.srl_lstm = nn.LSTM(H, 256, num_layers=1, bidirectional=True, batch_first=True)
model.srl_proj = nn.Linear(512, H)
model.srl_lstm.load_state_dict(state["srl_lstm"])
model.srl_proj.load_state_dict(state["srl_proj"])

# DEP head — load from state if exists, otherwise fresh
model.dep_scalar_mix = ScalarMix(num_layers, favor_range=(12, 18))
in_dim = H + len(pos_labels) + len(ner_labels)
model.dep_head = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, H), nn.GELU(), nn.Dropout(0.1), nn.Linear(H, len(dep_labels)))
if "dep_scalar_mix" in state:
    model.dep_scalar_mix.load_state_dict(state["dep_scalar_mix"])
    model.dep_head.load_state_dict(state["dep_head"])
    print("DEP head loaded from leader (0.743)")
else:
    print("DEP head fresh")

# CLS head — load if exists
model.cls_scalar_mix = ScalarMix(num_layers, favor_range=(18, 23))
model.cls_head = nn.Sequential(nn.LayerNorm(H), nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(0.1), nn.Linear(H // 2, len(cls_labels)))
if "cls_scalar_mix" in state:
    model.cls_scalar_mix.load_state_dict(state["cls_scalar_mix"])
    model.cls_head.load_state_dict(state["cls_head"])
    print("CLS head loaded from leader (0.525)")

model = model.to(device)

# Freeze all, unfreeze DEP only
for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "dep_scalar_mix" in n or "dep_head" in n:
        p.requires_grad = True
print(f"DEP trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def fwd_dep(input_ids, attention_mask):
    out = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    dep_h = model.dep_scalar_mix(out.hidden_states)
    pos_h = model.pos_scalar_mix(out.hidden_states)
    pos_probs = torch.softmax(model.pos_head(pos_h), dim=-1).detach()
    ner_h = model.ner_scalar_mix(out.hidden_states)
    ner_probs = torch.softmax(model.ner_head(torch.cat([ner_h, pos_probs], dim=-1)), dim=-1).detach()
    return model.dep_head(torch.cat([dep_h, pos_probs, ner_probs], dim=-1))

class TokenDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, label_key, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.label_key, self.max_length = examples, tokenizer, label_map, label_key, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        tags = ex[self.label_key]
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(tags[wid] if wid < len(tags) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}

with open("data/prepared/kniv-deberta-cascade/ud_train.json") as f: ud_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f: ud_dev = json.load(f)
dep_map = {l: i for i, l in enumerate(dep_labels)}
dep_train_loader = DataLoader(TokenDataset(ud_train, tokenizer, dep_map, "dep_labels"), batch_size=64, shuffle=True, collate_fn=collate)
dep_dev_loader = DataLoader(TokenDataset(ud_dev, tokenizer, dep_map, "dep_labels"), batch_size=64, collate_fn=collate)

dep_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(dep_params, lr=5e-4)
epochs = 20
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(dep_train_loader)*0.05), epochs*len(dep_train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

head_scores = {
    "pos": {"score": 0.966, "metric": "accuracy"},
    "ner": {"score": 0.846, "metric": "F1"},
    "srl": {"score": 0.802, "metric": "F1"},
    "dep": {"score": 0.743, "metric": "accuracy"},
    "cls": {"score": 0.525, "metric": "macro_f1"},
}
print(f"\nDEP continued: {epochs} epochs, LR=5e-4\n")

best_dep = 0.743
for epoch in range(epochs):
    model.train()
    tl = 0.0
    for batch in tqdm(dep_train_loader, desc=f"DEP E{epoch+1}"):
        logits = fwd_dep(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(dep_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        tl += loss.item()
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in dep_dev_loader:
            logits = fwd_dep(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            mask = batch["labels"] != -100
            correct += (preds[mask] == batch["labels"][mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total
    print(f"DEP E{epoch+1}/{epochs} — Loss: {tl/len(dep_train_loader):.4f}, Acc: {acc:.4f}")
    if acc > best_dep:
        best_dep = acc
        state["dep_scalar_mix"] = model.dep_scalar_mix.state_dict()
        state["dep_head"] = model.dep_head.state_dict()
        head_scores["dep"] = {"score": round(best_dep, 4), "metric": "accuracy"}
        save_leader(state, f"v2.2_dep{round(best_dep*1000)}", head_scores)

print(f"\nBest DEP Acc: {best_dep:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/kniv-nlp-models'

In [ ]:
# Cell 36: CRF only — freeze NER head + scalar mix, train ONLY CRF transitions
# The NER head already produces good emissions (0.687). CRF just learns valid transitions.
import json, torch, sys
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seq_f1
from torchcrf import CRF
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Reload best NER weights (undo the CRF damage)
ner_sm_state = torch.load(f"{DRIVE_OUT}/ner_scalarmix_best.pt", weights_only=True)
model.ner_scalar_mix.load_state_dict(ner_sm_state["ner_scalar_mix"])
model.ner_head_mix.load_state_dict(ner_sm_state["ner_head_mix"])

# Fresh CRF
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
ner_labels = vocabs["ner_labels"]
model.ner_crf = CRF(len(ner_labels), batch_first=True)
model = model.to(device)

# ONLY CRF is trainable — everything else frozen
for n, p in model.named_parameters():
    p.requires_grad = "ner_crf" in n
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (CRF transitions ONLY — {len(ner_labels)}x{len(ner_labels)} matrix)")

# Reuse forward_ner_emissions from before (frozen, produces emissions)
train_loader = DataLoader(NERDataset(ner_train, tokenizer, ner_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(NERDataset(ner_dev[:2000], tokenizer, ner_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-2)
epochs = 5
print(f"\nCRF-only training: {epochs} epochs\n")

best_f1 = 0.687
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        emissions = forward_ner_emissions(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        labels = batch["labels"].to(device)
        mask = labels != -100
        crf_labels = labels.clone()
        crf_labels[~mask] = 0
        loss = -model.ner_crf(emissions.detach(), crf_labels, mask=mask, reduction="mean")
        optimizer.zero_grad(); loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    all_gold, all_pred = [], []
    with torch.no_grad():
        for batch in dev_loader:
            emissions = forward_ner_emissions(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            labels = batch["labels"]
            mask = labels != -100
            decoded = model.ner_crf.decode(emissions, mask=mask.to(device))
            for j in range(labels.size(0)):
                gold_seq, pred_seq = [], []
                dec_idx = 0
                for k in range(labels.size(1)):
                    if labels[j, k] != -100:
                        gold_seq.append(ner_labels[labels[j, k]])
                        if dec_idx < len(decoded[j]):
                            pred_seq.append(ner_labels[decoded[j][dec_idx]] if decoded[j][dec_idx] < len(ner_labels) else "O")
                        else:
                            pred_seq.append("O")
                        dec_idx += 1
                all_gold.append(gold_seq)
                all_pred.append(pred_seq)
    f1 = seq_f1(all_gold, all_pred)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, NER F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.ner_crf.state_dict(), f"{DRIVE_OUT}/ner_crf_only_best.pt")
        print(f"  * New best F1={f1:.4f}")

print(f"\nBest NER F1: {best_f1:.4f}")

In [ ]:
# Cell 31: POS with Scalar Mix — load from pos_ner_srl_v1 (latest)
import json, torch, sys, os
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from train_srl_encoder import SRLEncoderModel, SRL_TAGS
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load latest model (pos_ner_srl_v1)
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/pos_ner_srl_v1")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
ner_labels = vocabs["ner_labels"]
H = model.deberta.config.hidden_size

model.pos_head = nn.Linear(H, len(pos_labels))
model.ner_head = nn.Sequential(
    nn.LayerNorm(H + len(pos_labels)),
    nn.Linear(H + len(pos_labels), H),
    nn.GELU(), nn.Dropout(0.1),
    nn.Linear(H, len(ner_labels)),
)

state = torch.load(f"{DRIVE_OUT}/pos_ner_srl_v1/model.pt", weights_only=True)
model.deberta.load_state_dict(state["deberta"])
model.pred_embedding.load_state_dict(state["pred_embedding"])
model.classifier.load_state_dict(state["classifier"])
model.pos_head.load_state_dict(state["pos_head"])
model.ner_head.load_state_dict(state["ner_head"])
print("Loaded pos_ner_srl_v1")

# Scalar mix
class ScalarMix(nn.Module):
    def __init__(self, num_layers):
        super().__init__()
        self.weights = nn.Parameter(torch.zeros(num_layers))
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, all_layers):
        normed = torch.softmax(self.weights, dim=0)
        mix = sum(w * layer for w, layer in zip(normed, all_layers))
        return self.scale * mix

num_layers = model.deberta.config.num_hidden_layers + 1
model.pos_scalar_mix = ScalarMix(num_layers)
# New POS head that reads from scalar mix (randomly initialized)
model.pos_head_mix = nn.Linear(H, len(pos_labels))
model = model.to(device)

# Freeze everything except scalar_mix + pos_head_mix
for n, p in model.named_parameters():
    p.requires_grad = "pos_scalar_mix" in n or "pos_head_mix" in n
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (scalar_mix + pos_head_mix)")

def forward_pos_mix(input_ids, attention_mask):
    outputs = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    mixed = model.pos_scalar_mix(outputs.hidden_states)
    return model.pos_head_mix(mixed)

class POSDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["pos_tags"][wid] if wid < len(ex["pos_tags"]) else "X", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
with open("data/prepared/kniv-deberta-cascade/ud_train.json") as f: ud_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f: ud_dev = json.load(f)
pos_map = {l: i for i, l in enumerate(pos_labels)}
train_loader = DataLoader(POSDataset(ud_train, tokenizer, pos_map), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(POSDataset(ud_dev, tokenizer, pos_map), batch_size=64, collate_fn=collate)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
epochs = 10
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(train_loader)*0.1), epochs*len(train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"\nPOS Scalar Mix: {epochs} epochs, {len(train_loader)} batches/epoch\n")

best_acc = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = forward_pos_mix(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(pos_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        total_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in dev_loader:
            logits = forward_pos_mix(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            mask = batch["labels"] != -100
            correct += (preds[mask] == batch["labels"][mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total
    w = torch.softmax(model.pos_scalar_mix.weights, dim=0)
    top3 = torch.topk(w, 3)
    layer_info = ", ".join(f"L{i}={v:.3f}" for i, v in zip(top3.indices.tolist(), top3.values.tolist()))
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, POS Acc: {acc:.4f} | Top layers: {layer_info}")
    if acc > best_acc:
        best_acc = acc
        torch.save({"pos_scalar_mix": model.pos_scalar_mix.state_dict(), "pos_head_mix": model.pos_head_mix.state_dict()},
                    f"{DRIVE_OUT}/pos_scalarmix_best.pt")
        print(f"  * New best Acc={acc:.4f}")

print(f"\nBest POS Acc: {best_acc:.4f}")
print(f"\nFinal layer weights:")
w = torch.softmax(model.pos_scalar_mix.weights, dim=0)
for i, v in enumerate(w.tolist()):
    bar = "█" * int(v * 100)
    if v > 0.02: print(f"  Layer {i:2d}: {v:.4f} {bar}")

In [ ]:
# Cell 20: Phase 2, Step 1 — Add POS head to gold-adapted SRL model
# Freeze encoder + pred_embedding + SRL classifier. Train only POS head.
import subprocess, os, json, torch, sys
os.chdir("/content/kniv-nlp-models")
subprocess.check_call(["git", "pull"])

sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import importlib, train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRLDataset, collate_fn, evaluate, SRL_TAGS, TAG2ID
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm
from seqeval.metrics import f1_score as seq_f1, classification_report

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"

# Load gold-adapted SRL model (0.738 Viterbi)
print("Loading gold-adapted SRL model...")
tokenizer = AutoTokenizer.from_pretrained(f"{DRIVE_OUT}/gold_best_v1_0738")
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{DRIVE_OUT}/gold_best_v1_0738/model.pt", weights_only=True)
model.load_state_dict(state)

# Add POS head
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
pos_labels = vocabs["pos_labels"]
H = model.deberta.config.hidden_size
model.pos_head = nn.Linear(H, len(pos_labels))
model = model.to(device)

# Freeze everything except pos_head
for name, param in model.named_parameters():
    if "pos_head" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (pos_head only)")

# Override forward to return POS logits
original_forward = model.forward
def forward_with_pos(input_ids, attention_mask, predicate_idx):
    B, S = input_ids.size()
    emb = model.deberta.embeddings(input_ids)
    indicator = torch.zeros(B, S, dtype=torch.long, device=input_ids.device)
    indicator[torch.arange(B, device=input_ids.device), predicate_idx] = 1
    emb = emb + model.pred_embedding(indicator)
    hidden = model.deberta.encoder(emb, attention_mask).last_hidden_state
    return model.pos_head(hidden)

# Load POS data
with open("data/prepared/kniv-deberta-cascade/ud_train.json") as f:
    ud_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f:
    ud_dev = json.load(f)
print(f"POS train: {len(ud_train):,}, dev: {len(ud_dev):,}")

# POS dataset — needs predicate_idx. Use dummy (0) since encoder is frozen
# and pred_embedding is frozen too — it just adds a fixed vector.
class POSDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples = examples
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_length = max_length
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        words = ex["words"]
        tags = ex["pos_tags"]
        encoding = self.tokenizer(words, is_split_into_words=True,
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt")
        word_ids = encoding.word_ids()
        aligned = []
        prev = None
        for wid in word_ids:
            if wid is None: aligned.append(-100)
            elif wid != prev:
                aligned.append(self.label_map.get(tags[wid] if wid < len(tags) else "X", 0))
            else: aligned.append(-100)
            prev = wid
        # Find first verb for predicate_idx (or 0 if none)
        pred_idx = 0
        for k, wid in enumerate(word_ids):
            if wid is not None and wid < len(ex.get("pos_tags", [])):
                if ex["pos_tags"][wid] == "VERB":
                    pred_idx = k
                    break
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(aligned, dtype=torch.long),
            "predicate_idx": torch.tensor(pred_idx, dtype=torch.long),
        }

pos_map = {l: i for i, l in enumerate(pos_labels)}
train_ds = POSDataset(ud_train, tokenizer, pos_map)
dev_ds = POSDataset(ud_dev, tokenizer, pos_map)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
    collate_fn=lambda feats: {
        "input_ids": torch.stack([f["input_ids"] for f in feats]),
        "attention_mask": torch.stack([f["attention_mask"] for f in feats]),
        "labels": torch.stack([f["labels"] for f in feats]),
        "predicate_idx": torch.stack([f["predicate_idx"] for f in feats]),
    })
dev_loader = DataLoader(dev_ds, batch_size=64, collate_fn=train_loader.collate_fn)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
epochs = 5
total_steps = epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*0.1), total_steps)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

print(f"\nPOS training: {epochs} epochs, {len(train_loader)} batches/epoch\n")

best_acc = 0.0
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        logits = forward_with_pos(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["predicate_idx"].to(device))
        loss = loss_fn(logits.view(-1, len(pos_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    # Eval
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in dev_loader:
            logits = forward_with_pos(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["predicate_idx"].to(device))
            preds = logits.argmax(dim=-1).cpu()
            labels = batch["labels"]
            mask = labels != -100
            correct += (preds[mask] == labels[mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, POS Acc: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.pos_head.state_dict(), f"{DRIVE_OUT}/pos_head_best.pt")
        print(f"  * New best Acc={acc:.4f}")

print(f"\nDone. Best POS Acc: {best_acc:.4f}")